# Regime Robustness Sandbox Notebook
This notebook reuses the full `project.ipynb` pipeline inside the sandbox only. It downloads or rebuilds sandbox-local raw data, filters the experiment to the 2020--2024 window, and writes all generated artifacts under this sandbox so the main project remains untouched.


**How to use this notebook**

1. Use the same Python environment as the main project and make sure `requests` is installed.
2. Run the notebook from top to bottom.
3. All outputs stay inside this sandbox's `data/`, `models/`, and `result/` folders.


## 1. Scope and Experiment Design
This sandbox keeps the full project workflow intact while changing only two things: the raw-data acquisition step and the active sample window. Everything else (feature engineering, split logic, models, thresholding, backtests, tables, and plots) follows the main notebook.


In [ ]:
import os
os.environ.setdefault('TF_DETERMINISTIC_OPS', '1')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')
import json
import gc
import random
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
from itertools import product
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

import io
import zipfile
try:
    import requests
except ImportError as exc:
    raise ImportError("This sandbox notebook needs the `requests` package for Binance raw-data downloads. Install the main repo requirements and then `pip install requests` in the same environment.") from exc
from IPython.display import display

In [ ]:
# -- Sandbox execution switches ------------------------------------
FORCE_DOWNLOAD_RAW = False
FORCE_REBUILD_RAW = False
FORCE_DATA_PREP = True  # regenerate engineered features and chronological splits from sandbox raw CSV files
RUN_UNIFIED_GRID = True   # run the main experiment grid that drives all downstream tables and plots

# -- Sample-window controls ----------------------------------------
RUN_LABEL = 'shifted_2020_2024'
SAMPLE_START = '2020-01-01 00:00:00+00:00'
SAMPLE_END = '2024-12-31 23:00:00+00:00'

# -- Raw-download controls -----------------------------------------
DOWNLOAD_START = '2020-01'
DOWNLOAD_END = '2024-12'
REQUEST_TIMEOUT = 60
GLOBAL_SEED = 42
PRINT_DEBUG_MESSAGES = False
PRINT_MODEL_TRAINING_MESSAGES = False
LOGISTIC_MODEL_VERBOSE = 1 if PRINT_MODEL_TRAINING_MESSAGES else 0
XGBOOST_MODEL_VERBOSITY = 2 if PRINT_MODEL_TRAINING_MESSAGES else 0
LIGHTGBM_MODEL_VERBOSITY = 1 if PRINT_MODEL_TRAINING_MESSAGES else -1
SEQUENCE_MODEL_FIT_VERBOSE = 1 if PRINT_MODEL_TRAINING_MESSAGES else 0

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
keras.utils.set_random_seed(GLOBAL_SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    print("Fail to enable deterministic ops")

def debug_print(*args, **kwargs):
    if PRINT_DEBUG_MESSAGES:
        print('[DEBUG]', *args, **kwargs)


TREE_MODEL_N_JOBS = 1

# -- Paths ----------------------------------------------------------
SANDBOX_ROOT = Path.cwd().resolve()
if SANDBOX_ROOT.name != '2020-2024_regime_sandbox':
    SANDBOX_ROOT = SANDBOX_ROOT / '2020-2024_regime_sandbox'
if not (SANDBOX_ROOT / 'regime_shift_sandbox.ipynb').exists():
    raise FileNotFoundError('Could not locate regime_shift_sandbox.ipynb under the current directory.')
SANDBOX_DIR = SANDBOX_ROOT
ROOT_DIR = SANDBOX_ROOT
print(f'Sandbox root resolved to: {ROOT_DIR}')
DATA_RAW_DIR = ROOT_DIR / 'data' / 'raw'
DATA_FEAT_DIR = ROOT_DIR / 'data' / 'engineered_features'
DATA_SPLIT_DIR = ROOT_DIR / 'data' / 'split_data'
RESULT_DIR = ROOT_DIR / 'result'
MODELS_DIR = ROOT_DIR / 'models'
DATA_LOG_DIR = ROOT_DIR / 'data' / 'trading_logs'

for d in [
    DATA_FEAT_DIR,
    DATA_SPLIT_DIR,
    MODELS_DIR,
    RESULT_DIR / 'buy_and_hold',
    RESULT_DIR / 'paper_tables',
    RESULT_DIR / 'stat_tests',
    RESULT_DIR / 'graph',
    DATA_LOG_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# -- Experiment constants ----------------------------------
ASSETS = ['BTC', 'ETH', 'XRP', 'SOL']
TRANSACTION_COST = 0.00035
PERIODS_PER_YEAR = 8760
EPS = 1e-12
LOOKBACK_HOURS = 24
LONG_T_GRID = np.arange(0.50, 0.75, 0.01)
SHORT_T_GRID = np.arange(0.50, 0.75, 0.01)
GRU_LONG_T_GRID = np.arange(0.47, 0.75, 0.01)
GRU_SHORT_T_GRID = np.arange(0.47, 0.75, 0.01)
GRU_BATCH_SIZE = 256
GRU_EARLY_STOPPING_PATIENCE = 20

SYMBOLS = {
    'BTC': 'BTCUSDT',
    'ETH': 'ETHUSDT',
    'XRP': 'XRPUSDT',
    'SOL': 'SOLUSDT',
}
INPUT_FILES = {asset: str(DATA_RAW_DIR / f'{symbol}_1h.csv') for asset, symbol in SYMBOLS.items()}

TARGET_LONG_COL = 'label_up'
TARGET_SHORT_COL = 'label_down'
RETURN_COL = 'fwd_ret_open_1h'
LOG_RETURN_COL = 'fwd_log_ret_open_1h'

PAPER_CYCLICAL_FEATURES = ['hour_sin', 'hour_cos']
PAPER_CANDLE_FEATURES = ['close', 'candle_body', 'upper_shadow', 'lower_shadow']
PAPER_OHLC_FEATURES = ['open', 'high', 'low', 'close']
PAPER_TECH_IND_FEATURES = [
    'sma_15', 'sma_20', 'sma_25', 'sma_30',
    'rsi_15', 'rsi_20', 'rsi_25', 'rsi_30',
    'macd_line', 'macd_hist', 'wr_14', 'so_14', 'mfi_14',
]

OHLC_RAW_FEATURES = PAPER_OHLC_FEATURES + ['number_of_trades', 'volume'] + PAPER_CYCLICAL_FEATURES
CANDLE_RAW_FEATURES = PAPER_CANDLE_FEATURES + ['number_of_trades', 'volume'] + PAPER_CYCLICAL_FEATURES
TECH_IND_FEATURES = PAPER_TECH_IND_FEATURES
FEATURE_SET_DEFS = {
    'ohlc_raw': OHLC_RAW_FEATURES,
    'candle_raw': CANDLE_RAW_FEATURES,
    'ohlc_extended': OHLC_RAW_FEATURES + TECH_IND_FEATURES,
    'candle_extended': CANDLE_RAW_FEATURES + TECH_IND_FEATURES,
}

EXTRA_FEATURE_COLS = [
    'quote_asset_volume', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
    'ret_1h_close', 'log_ret_1h_close', 'ret_3h', 'log_ret_3h', 'ret_6h', 'log_ret_6h',
    'ret_12h', 'log_ret_12h', 'ret_24h', 'log_ret_24h', 'ret_48h', 'log_ret_48h',
    'ret_72h', 'log_ret_72h', 'body_pct', 'body_abs_pct', 'range_pct', 'body_to_range',
    'close_loc', 'bar_up', 'bar_down', 'gap_open_prev_close', 'close_to_sma_6',
    'close_to_sma_12', 'close_to_sma_24', 'close_to_sma_48', 'close_to_sma_72',
    'close_to_ema_6', 'close_to_ema_12', 'close_to_ema_24', 'close_to_ema_48',
    'close_to_ema_72', 'sma_cross_6_24', 'sma_cross_12_48', 'ema_cross_12_24',
    'vol_6h', 'vol_24h', 'vol_72h', 'downside_vol_24h', 'downside_vol_72h',
    'atr_14_pct', 'macd_signal', 'macd_signal_pct', 'macd_line_pct', 'macd_hist_pct',
    'log_volume', 'log_quote_volume', 'log_trades', 'vol_z_24', 'quote_vol_z_24',
    'trades_z_24', 'vol_ratio_24', 'quote_vol_ratio_24', 'taker_buy_base_share',
    'taker_buy_quote_share', 'order_flow_imbalance', 'dow_sin', 'dow_cos',
]
FEATURE_COLS = sorted(set(sum(FEATURE_SET_DEFS.values(), [])) | set(EXTRA_FEATURE_COLS))
FULL_EXTENDED_FEATURES = sorted(set(FEATURE_COLS))

UNIFIED_MODELS = ['Logistic Regression', 'XGBoost', 'LightGBM', 'GRU', 'SMA', 'RSI']
ML_MODELS = ['Logistic Regression', 'XGBoost', 'LightGBM', 'GRU']
TABULAR_ML_MODELS = ['Logistic Regression', 'XGBoost', 'LightGBM']
SEQUENCE_ML_MODELS = ['GRU']
RULE_MODELS = ['SMA', 'RSI']
UNIFIED_FEATURE_SETS = ['ohlc_raw', 'candle_raw', 'ohlc_extended', 'candle_extended']
UNIFIED_STRATEGY_MODES = ['long_only', 'short_only', 'long_short']
COST_REGIMES = {'with_cost': TRANSACTION_COST, 'no_cost': 0.0}

SMA_SHORT_WINDOWS = [5, 10, 20, 30]
SMA_LONG_WINDOWS = [50, 100, 150, 200]
RSI_WINDOWS = [7, 14, 21]

MIN_TRADES_VALID = 20
MIN_SIGNAL_RATIO = 0.002
MIN_PRECISION_RECALL = 0.01

print(f'Running sandbox pipeline for: {RUN_LABEL}')
print(f'Sample window: {SAMPLE_START} -> {SAMPLE_END}')
print(f'Output root: {ROOT_DIR}')

Sandbox root resolved to: /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox
Running sandbox pipeline for: shifted_2020_2024
Sample window: 2020-01-01 00:00:00+00:00 -> 2024-12-31 23:00:00+00:00
Output root: /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox


## 1A. Raw Data Import
This sandbox-only step downloads monthly Binance spot 1-hour klines for 2020--2024, caches them under sandbox `data/raw/monthly/`, and rebuilds the merged raw CSVs used by the main pipeline.


In [ ]:

# -- Raw monthly data download and merged-csv refresh -----------------
BINANCE_MONTHLY_URL = 'https://data.binance.vision/data/spot/monthly/klines/{symbol}/1h/{symbol}-1h-{year}-{month:02d}.zip'
MONTHLY_RAW_DIR = DATA_RAW_DIR / 'monthly'
KLINE_COLUMNS = [
    'open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume',
    'taker_buy_quote_asset_volume', 'ignore',
]
NUMERIC_COLUMNS = [
    'open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_asset_volume', 'number_of_trades', 'taker_buy_base_asset_volume',
    'taker_buy_quote_asset_volume',
]


def _iter_months(start_ym: str, end_ym: str):
    for period in pd.period_range(start=start_ym, end=end_ym, freq='M'):
        yield int(period.year), int(period.month)


def _monthly_csv_path(symbol: str, year: int, month: int) -> Path:
    return MONTHLY_RAW_DIR / symbol / f'{symbol}-1h-{year}-{month:02d}.csv'


def _download_month(symbol: str, year: int, month: int):
    out_path = _monthly_csv_path(symbol, year, month)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and not FORCE_DOWNLOAD_RAW:
        return out_path, 'cached'

    url = BINANCE_MONTHLY_URL.format(symbol=symbol, year=year, month=month)
    response = requests.get(url, timeout=REQUEST_TIMEOUT)
    if response.status_code == 404:
        return None, 'missing'
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        csv_members = [name for name in zf.namelist() if name.endswith('.csv')]
        if not csv_members:
            raise ValueError(f'No CSV payload found in {url}')
        payload = zf.read(csv_members[0])
    out_path.write_bytes(payload)
    return out_path, 'downloaded'


def _load_month_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=None)
    if len(df.columns) >= len(KLINE_COLUMNS):
        df = df.iloc[:, :len(KLINE_COLUMNS)]
    df.columns = KLINE_COLUMNS[:len(df.columns)]
    if str(df.iloc[0, 0]).strip().lower() == 'open_time':
        df = df.iloc[1:].copy()
    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['open_time', 'open', 'high', 'low', 'close', 'volume']).reset_index(drop=True)
    return df


def _rebuild_symbol_raw(symbol: str) -> pd.DataFrame:
    monthly_dir = MONTHLY_RAW_DIR / symbol
    monthly_files = sorted(monthly_dir.glob(f'{symbol}-1h-*.csv'))
    if len(monthly_files) == 0:
        raise FileNotFoundError(f'No monthly raw files available for {symbol}.')
    frames = [_load_month_csv(path) for path in monthly_files]
    merged = pd.concat(frames, ignore_index=True)
    merged = merged.sort_values('open_time').drop_duplicates(subset='open_time', keep='last').reset_index(drop=True)
    merged_path = DATA_RAW_DIR / f'{symbol}_1h.csv'
    merged.to_csv(merged_path, index=False)
    return merged


download_rows = []
for asset, symbol in SYMBOLS.items():
    for year, month in _iter_months(DOWNLOAD_START, DOWNLOAD_END):
        try:
            path, status = _download_month(symbol, year, month)
        except Exception as exc:
            status = f'error: {exc}'
            path = None
        download_rows.append({
            'asset': asset,
            'symbol': symbol,
            'year': year,
            'month': month,
            'status': status,
            'path': str(path) if path is not None else '',
        })

download_summary = pd.DataFrame(download_rows)
download_summary.to_csv(DATA_RAW_DIR / 'download_status.csv', index=False)
print('Raw monthly download status saved ->', DATA_RAW_DIR / 'download_status.csv')

total_missing = int((download_summary['status'] == 'missing').sum())
if total_missing > 0:
    print(f'Note: {total_missing} monthly files were unavailable from Binance and were skipped.')

coverage_rows = []
for asset, symbol in SYMBOLS.items():
    merged_path = DATA_RAW_DIR / f'{symbol}_1h.csv'
    if not merged_path.exists() or FORCE_REBUILD_RAW:
        merged = _rebuild_symbol_raw(symbol)
    else:
        merged = pd.read_csv(merged_path)
    open_time = pd.to_numeric(merged['open_time'], errors='coerce')
    dt = pd.Series(pd.NaT, index=open_time.index, dtype='datetime64[ns, UTC]')
    us_mask = open_time >= 10**15
    ms_mask = open_time.between(10**12, 10**15 - 1, inclusive='both')
    dt.loc[us_mask] = pd.to_datetime(open_time.loc[us_mask], unit='us', utc=True, errors='coerce')
    dt.loc[ms_mask] = pd.to_datetime(open_time.loc[ms_mask], unit='ms', utc=True, errors='coerce')
    dt = dt.dropna().sort_values()
    coverage_rows.append({
        'asset': asset,
        'symbol': symbol,
        'rows': int(len(merged)),
        'start': dt.iloc[0] if len(dt) else pd.NaT,
        'end': dt.iloc[-1] if len(dt) else pd.NaT,
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(DATA_RAW_DIR / 'raw_coverage_summary.csv', index=False)
print('Raw coverage summary saved ->', DATA_RAW_DIR / 'raw_coverage_summary.csv')
display(coverage_df)

Raw monthly download status saved -> /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/data/raw/download_status.csv
Note: 7 monthly files were unavailable from Binance and were skipped.
Raw coverage summary saved -> /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/data/raw/raw_coverage_summary.csv


,asset,symbol,rows,start,end
0,BTC,BTCUSDT,43817,2020-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00
1,ETH,ETHUSDT,43817,2020-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00
2,XRP,XRPUSDT,43817,2020-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00
3,SOL,SOLUSDT,38471,2020-08-11 06:00:00+00:00,2024-12-31 23:00:00+00:00


## 2. Data Preparation and Paper-Aligned Feature Engineering
The feature-engineering stage is identical to the main project pipeline, with one sandbox-specific addition: feature rows are filtered to the selected 2020--2024 sample window before chronological splitting.


In [ ]:
# -- Indicator helpers ----------------------------------
def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    rs = avg_gain / (avg_loss + EPS)
    return 100 - (100 / (1 + rs))


def compute_macd(close, fast=12, slow=26, signal=9):
    ema_f = close.ewm(span=fast, adjust=False, min_periods=fast).mean()
    ema_s = close.ewm(span=slow, adjust=False, min_periods=slow).mean()
    line = ema_f - ema_s
    sig = line.ewm(span=signal, adjust=False, min_periods=signal).mean()
    return line, sig, line - sig


def compute_atr(high, low, close, window=14):
    pc = close.shift(1)
    tr = pd.concat([high - low, (high - pc).abs(), (low - pc).abs()], axis=1).max(axis=1)
    return tr.rolling(window, min_periods=window).mean()


def compute_williams_r(high, low, close, window=14):
    hh = high.rolling(window, min_periods=window).max()
    ll = low.rolling(window, min_periods=window).min()
    return -100 * (hh - close) / (hh - ll + EPS)


def compute_stochastic_oscillator(high, low, close, window=14):
    hh = high.rolling(window, min_periods=window).max()
    ll = low.rolling(window, min_periods=window).min()
    k = 100 * (close - ll) / (hh - ll + EPS)
    d = k.rolling(3, min_periods=3).mean()
    return k - d


def compute_mfi(high, low, close, volume, window=14):
    tp = (high + low + close) / 3
    rmf = tp * volume
    tp_diff = tp.diff()
    pos = rmf.where(tp_diff > 0, 0.0)
    neg = rmf.where(tp_diff < 0, 0.0).abs()
    pmf = pos.rolling(window, min_periods=window).sum()
    nmf = neg.rolling(window, min_periods=window).sum()
    mfr = pmf / (nmf + EPS)
    return 100 - 100 / (1 + mfr)


def rolling_zscore(series, window):
    m = series.rolling(window, min_periods=window).mean()
    s = series.rolling(window, min_periods=window).std()
    return (series - m) / (s + EPS)

In [ ]:
# -- Raw-data loading -----------------------------------
def load_and_standardize(file_path: str, asset_name: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    keep = [
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'quote_asset_volume', 'number_of_trades',
        'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
    ]
    df = df[keep].copy()
    df['open_time'] = pd.to_numeric(df['open_time'], errors='coerce')
    # Binance exports can mix millisecond and microsecond timestamps in the same file.
    # Detect the unit row-wise instead of trying one unit and falling back to the other,
    # which would silently interpret microsecond values as far-future millisecond dates.
    ms_mask = df['open_time'].abs() < 10 ** 14
    dt = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns, UTC]')
    dt.loc[ms_mask] = pd.to_datetime(df.loc[ms_mask, 'open_time'], unit='ms', utc=True, errors='coerce')
    dt.loc[~ms_mask] = pd.to_datetime(df.loc[~ms_mask, 'open_time'], unit='us', utc=True, errors='coerce')
    df['datetime'] = dt
    df = df.dropna(subset=['datetime']).sort_values('datetime').reset_index(drop=True)
    for c in [
        'open', 'high', 'low', 'close', 'volume', 'quote_asset_volume',
        'number_of_trades', 'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
    ]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.drop_duplicates('datetime').dropna(subset=['open', 'high', 'low', 'close'])
    df = df[(df['open'] > 0) & (df['high'] > 0) & (df['low'] > 0) & (df['close'] > 0)].copy()
    df['asset'] = asset_name
    return df.reset_index(drop=True)

In [ ]:
# -- Feature construction -------------------------------
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out['log_close'] = np.log(out['close'])
    out['ret_1h_close'] = out['close'].pct_change(1)
    out['log_ret_1h_close'] = np.log(out['close'] / out['close'].shift(1))
    for h in [3, 6, 12, 24, 48, 72]:
        out[f'ret_{h}h'] = out['close'].pct_change(h)
        out[f'log_ret_{h}h'] = np.log(out['close'] / out['close'].shift(h))

    max_oc = np.maximum(out['open'], out['close'])
    min_oc = np.minimum(out['open'], out['close'])
    out['range_pct'] = (out['high'] - out['low']) / (out['open'] + EPS)
    out['body_pct'] = (out['close'] - out['open']) / (out['open'] + EPS)
    out['body_abs_pct'] = out['body_pct'].abs()
    out['upper_shadow_pct'] = (out['high'] - max_oc) / (out['open'] + EPS)
    out['lower_shadow_pct'] = (min_oc - out['low']) / (out['open'] + EPS)
    out['body_to_range'] = (out['close'] - out['open']) / ((out['high'] - out['low']) + EPS)
    out['close_loc'] = (out['close'] - out['low']) / ((out['high'] - out['low']) + EPS)
    out['bar_up'] = (out['close'] > out['open']).astype(int)
    out['bar_down'] = (out['close'] < out['open']).astype(int)
    out['gap_open_prev_close'] = out['open'] / out['close'].shift(1) - 1

    out['candle_body'] = (out['close'] - out['open']).abs()
    out['upper_shadow'] = np.where(out['close'] >= out['open'], out['high'] - out['close'], out['high'] - out['open'])
    out['lower_shadow'] = np.where(out['close'] >= out['open'], out['open'] - out['low'], out['close'] - out['low'])

    for w in [6, 12, 24, 48, 72]:
        out[f'sma_{w}'] = out['close'].rolling(w, min_periods=w).mean()
        out[f'ema_{w}'] = out['close'].ewm(span=w, adjust=False, min_periods=w).mean()
        out[f'close_to_sma_{w}'] = out['close'] / (out[f'sma_{w}'] + EPS) - 1
        out[f'close_to_ema_{w}'] = out['close'] / (out[f'ema_{w}'] + EPS) - 1
    out['sma_cross_6_24'] = out['sma_6'] / (out['sma_24'] + EPS) - 1
    out['sma_cross_12_48'] = out['sma_12'] / (out['sma_48'] + EPS) - 1
    out['ema_cross_12_24'] = out['ema_12'] / (out['ema_24'] + EPS) - 1

    for w in [15, 20, 25, 30]:
        out[f'sma_{w}'] = out['close'].rolling(w, min_periods=w).mean()
        out[f'rsi_{w}'] = compute_rsi(out['close'], w)

    for w in [6, 24, 72]:
        out[f'vol_{w}h'] = out['log_ret_1h_close'].rolling(w, min_periods=w).std()
    neg = out['log_ret_1h_close'].where(out['log_ret_1h_close'] < 0, 0)
    for w in [24, 72]:
        out[f'downside_vol_{w}h'] = neg.rolling(w, min_periods=w).std()
    out['atr_14'] = compute_atr(out['high'], out['low'], out['close'], 14)
    out['atr_14_pct'] = out['atr_14'] / (out['close'] + EPS)

    out['rsi_14'] = compute_rsi(out['close'], 14)
    macd_line, macd_signal, macd_hist = compute_macd(out['close'])
    out['macd_line'] = macd_line
    out['macd_signal'] = macd_signal
    out['macd_hist'] = macd_hist
    out['macd_line_pct'] = macd_line / (out['close'] + EPS)
    out['macd_signal_pct'] = macd_signal / (out['close'] + EPS)
    out['macd_hist_pct'] = macd_hist / (out['close'] + EPS)
    out['wr_14'] = compute_williams_r(out['high'], out['low'], out['close'], 14)
    out['so_14'] = compute_stochastic_oscillator(out['high'], out['low'], out['close'], 14)
    out['mfi_14'] = compute_mfi(out['high'], out['low'], out['close'], out['volume'], 14)

    out['log_volume'] = np.log1p(out['volume'])
    out['log_quote_volume'] = np.log1p(out['quote_asset_volume'])
    out['log_trades'] = np.log1p(out['number_of_trades'])
    out['vol_z_24'] = rolling_zscore(out['log_volume'], 24)
    out['quote_vol_z_24'] = rolling_zscore(out['log_quote_volume'], 24)
    out['trades_z_24'] = rolling_zscore(out['log_trades'], 24)
    out['vol_ratio_24'] = out['volume'] / (out['volume'].rolling(24, min_periods=24).mean() + EPS)
    out['quote_vol_ratio_24'] = out['quote_asset_volume'] / (out['quote_asset_volume'].rolling(24, min_periods=24).mean() + EPS)
    out['taker_buy_base_share'] = out['taker_buy_base_asset_volume'] / (out['volume'] + EPS)
    out['taker_buy_quote_share'] = out['taker_buy_quote_asset_volume'] / (out['quote_asset_volume'] + EPS)
    out['order_flow_imbalance'] = 2 * out['taker_buy_base_share'] - 1

    # Defragment before the final time/target columns so pandas does not warn
    # about the repeated feature-column insertions above.
    out = out.copy()

    out['hour'] = out['datetime'].dt.hour
    out['dayofweek'] = out['datetime'].dt.dayofweek
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow_sin'] = np.sin(2 * np.pi * out['dayofweek'] / 7)
    out['dow_cos'] = np.cos(2 * np.pi * out['dayofweek'] / 7)

    out['entry_open_t1'] = out['open'].shift(-1)
    out['exit_open_t2'] = out['open'].shift(-2)
    out['fwd_ret_open_1h'] = out['exit_open_t2'] / out['entry_open_t1'] - 1
    out['fwd_log_ret_open_1h'] = np.log(out['exit_open_t2'] / out['entry_open_t1'])
    out['label_up'] = (out['fwd_log_ret_open_1h'] > 0).astype(int)
    out['label_down'] = (out['fwd_log_ret_open_1h'] < 0).astype(int)

    out = out.replace([np.inf, -np.inf], np.nan).copy()
    return out

In [ ]:
# -- Feature-dataset assembly and execution -------------
def build_feature_dataset(file_path: str, asset_name: str, sample_start=None, sample_end=None) -> pd.DataFrame:
    raw = load_and_standardize(file_path, asset_name)
    feat = create_features(raw)
    keep_cols = [
        'datetime', 'asset', 'open', 'high', 'low', 'close', 'volume',
        'quote_asset_volume', 'number_of_trades',
        'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
        'entry_open_t1', 'exit_open_t2', RETURN_COL, LOG_RETURN_COL,
        TARGET_LONG_COL, TARGET_SHORT_COL,
        'candle_body', 'upper_shadow', 'lower_shadow',
        'ret_1h_close', 'log_ret_1h_close', 'ret_3h', 'log_ret_3h', 'ret_6h', 'log_ret_6h',
        'ret_12h', 'log_ret_12h', 'ret_24h', 'log_ret_24h', 'ret_48h', 'log_ret_48h',
        'ret_72h', 'log_ret_72h', 'range_pct', 'body_pct', 'body_abs_pct', 'upper_shadow_pct',
        'lower_shadow_pct', 'body_to_range', 'close_loc', 'bar_up', 'bar_down',
        'gap_open_prev_close', 'sma_6', 'sma_12', 'sma_15', 'sma_20', 'sma_24', 'sma_25',
        'sma_30', 'sma_48', 'sma_72', 'ema_6', 'ema_12', 'ema_24', 'ema_48', 'ema_72',
        'close_to_sma_6', 'close_to_sma_12', 'close_to_sma_24', 'close_to_sma_48', 'close_to_sma_72',
        'close_to_ema_6', 'close_to_ema_12', 'close_to_ema_24', 'close_to_ema_48', 'close_to_ema_72',
        'sma_cross_6_24', 'sma_cross_12_48', 'ema_cross_12_24',
        'rsi_14', 'rsi_15', 'rsi_20', 'rsi_25', 'rsi_30',
        'macd_line', 'macd_signal', 'macd_hist', 'macd_line_pct', 'macd_signal_pct', 'macd_hist_pct',
        'wr_14', 'so_14', 'mfi_14', 'vol_6h', 'vol_24h', 'vol_72h', 'downside_vol_24h',
        'downside_vol_72h', 'atr_14', 'atr_14_pct', 'log_volume', 'log_quote_volume', 'log_trades',
        'vol_z_24', 'quote_vol_z_24', 'trades_z_24', 'vol_ratio_24', 'quote_vol_ratio_24',
        'taker_buy_base_share', 'taker_buy_quote_share', 'order_flow_imbalance',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    ]
    base_required = [
        'datetime', 'asset', 'open', 'high', 'low', 'close', 'volume',
        'quote_asset_volume', 'number_of_trades',
        'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume',
        'entry_open_t1', 'exit_open_t2', RETURN_COL, LOG_RETURN_COL,
        TARGET_LONG_COL, TARGET_SHORT_COL,
        'candle_body', 'upper_shadow', 'lower_shadow',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    ]
    feat = feat[keep_cols]

    feat = feat.dropna(subset=base_required).reset_index(drop=True)
    if sample_start is not None:
        feat = feat[feat['datetime'] >= pd.Timestamp(sample_start, tz='UTC')].copy()
    if sample_end is not None:
        feat = feat[feat['datetime'] <= pd.Timestamp(sample_end, tz='UTC')].copy()
    feat = feat.reset_index(drop=True)
    if len(feat) == 0:
        raise ValueError(f'No feature rows remain for {asset_name} inside the selected sample window.')
    return feat


feat_files_exist = all((DATA_FEAT_DIR / f'{a}_features.csv').exists() for a in ASSETS)

if not FORCE_DATA_PREP and feat_files_exist:
    print('Engineered feature files found - skipping feature engineering.')
    print('Set FORCE_DATA_PREP = True to regenerate.')
else:
    print('Running feature engineering...')
    for asset, file_path in INPUT_FILES.items():
        df_feat = build_feature_dataset(file_path, asset, SAMPLE_START, SAMPLE_END)
        out_path = DATA_FEAT_DIR / f'{asset}_features.csv'
        df_feat.to_csv(out_path, index=False)
        print(f"  {asset}: {len(df_feat):,} rows | {df_feat['datetime'].min()} -> {df_feat['datetime'].max()} -> {out_path}")

Running feature engineering...
  BTC: 43,815 rows | 2020-01-01 00:00:00+00:00 -> 2024-12-31 21:00:00+00:00 -> /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/data/engineered_features/BTC_features.csv
  ETH: 43,815 rows | 2020-01-01 00:00:00+00:00 -> 2024-12-31 21:00:00+00:00 -> /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/data/engineered_features/ETH_features.csv
  XRP: 43,815 rows | 2020-01-01 00:00:00+00:00 -> 2024-12-31 21:00:00+00:00 -> /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/data/engineered_features/XRP_features.csv
  SOL: 38,469 rows | 2020-08-11 06:00:00+00:00 -> 2024-12-31 21:00:00+00:00 -> /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/data/engineered_features/SOL_features.csv


## 3. Chronological Train / Validation / Test Split
The split is strictly time ordered:
- 50% training,
- 25% validation,
- 25% test.

No random reshuffling is used. This keeps threshold tuning, hyperparameter assumptions, and out-of-sample evaluation chronologically honest.


In [ ]:
def chronological_split(df, train_frac=0.50, val_frac=0.25, test_frac=0.25, label_horizon_rows=2):
    if min(train_frac, val_frac, test_frac) <= 0:
        raise ValueError('train_frac, val_frac, and test_frac must all be positive.')
    if not np.isclose(train_frac + val_frac + test_frac, 1.0, atol=1e-8):
        raise ValueError('train_frac + val_frac + test_frac must sum to 1.')

    df = df.sort_values('datetime').reset_index(drop=True)
    n = len(df)
    t1 = int(n * train_frac)
    t2 = int(n * (train_frac + val_frac))

    train_df = df.iloc[:t1].copy()
    val_df = df.iloc[t1:t2].copy()
    test_df = df.iloc[t2:].copy()

    # Labels use future open prices. Trim train/validation tails so their
    # targets never depend on observations from the next split.
    if label_horizon_rows > 0:
        train_df = train_df.iloc[:-label_horizon_rows].copy() if len(train_df) > label_horizon_rows else train_df.iloc[:0].copy()
        val_df = val_df.iloc[:-label_horizon_rows].copy() if len(val_df) > label_horizon_rows else val_df.iloc[:0].copy()

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise ValueError('Chronological split produced an empty train, validation, or test block.')

    check_cols = [c for c in [RETURN_COL, LOG_RETURN_COL, TARGET_LONG_COL, TARGET_SHORT_COL] if c in test_df.columns]
    if check_cols and test_df[check_cols].isna().any().any():
        raise ValueError(f'Test split contains missing values in required forward-label columns: {check_cols}')

    return train_df, val_df, test_df

In [ ]:
split_files_exist = all(
    (DATA_SPLIT_DIR / f'{a}_{s}.csv').exists()
    for a in ASSETS for s in ['train', 'val', 'test']
)

if not FORCE_DATA_PREP and split_files_exist:
    print('Split data files found - skipping split.')
    print('Set FORCE_DATA_PREP = True to re-split.')
else:
    print('Running train/val/test split...')
    for asset in ASSETS:
        df = pd.read_csv(DATA_FEAT_DIR / f'{asset}_features.csv')
        df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce')
        train_df, val_df, test_df = chronological_split(df)
        for split, sdf in [('train', train_df), ('val', val_df), ('test', test_df)]:
            sdf.to_csv(DATA_SPLIT_DIR / f'{asset}_{split}.csv', index=False)
        print(f'  {asset}: train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}')

def load_split_csv(path):
    df = pd.read_csv(path, low_memory=False)
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce')
    return df


train_data, val_data, test_data = {}, {}, {}
for asset in ASSETS:
    train_data[asset] = load_split_csv(DATA_SPLIT_DIR / f'{asset}_train.csv')
    val_data[asset] = load_split_csv(DATA_SPLIT_DIR / f'{asset}_val.csv')
    test_data[asset] = load_split_csv(DATA_SPLIT_DIR / f'{asset}_test.csv')

split_summary_rows = []
for asset in ASSETS:
    split_summary_rows.append({
        'asset': asset,
        'total_rows': len(train_data[asset]) + len(val_data[asset]) + len(test_data[asset]),
        'train_rows': len(train_data[asset]),
        'val_rows': len(val_data[asset]),
        'test_rows': len(test_data[asset]),
        'train_start': train_data[asset]['datetime'].min(),
        'train_end': train_data[asset]['datetime'].max(),
        'val_start': val_data[asset]['datetime'].min(),
        'val_end': val_data[asset]['datetime'].max(),
        'test_start': test_data[asset]['datetime'].min(),
        'test_end': test_data[asset]['datetime'].max(),
    })
split_summary = pd.DataFrame(split_summary_rows)
split_summary.to_csv(DATA_SPLIT_DIR / 'split_summary.csv', index=False)

print('Data loaded into train_data / val_data / test_data.')
print(f"Split summary refreshed -> {DATA_SPLIT_DIR / 'split_summary.csv'}")

Running train/val/test split...
  BTC: train=21,905  val=10,952  test=10,954
  ETH: train=21,905  val=10,952  test=10,954
  XRP: train=21,905  val=10,952  test=10,954
  SOL: train=19,232  val=9,615  test=9,618
Data loaded into train_data / val_data / test_data.
Split summary refreshed -> /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/data/split_data/split_summary.csv


## 4. Shared Metrics, Threshold Selection, and Bootstrap Helpers
This section centralizes the evaluation logic used throughout the notebook.

Key rules:
- validation thresholds are chosen by precision first,
- candidate thresholds must still satisfy minimum trade-count, signal-ratio, and recall guardrails,
- the paper-style Sharpe in the summary tables is computed from signal returns rather than annualized hourly returns,
- grouped bootstrap tests follow the paper's non-overlapping block idea with block length `floor(n^(1/3))`.


In [ ]:
# -- Return and risk summary helpers --------------------
def compute_cumulative_return(returns):
    returns = pd.Series(returns).dropna()
    return (1 + returns).prod() - 1 if len(returns) > 0 else np.nan


def compute_annualized_return(returns, periods_per_year=8760):
    returns = pd.Series(returns).dropna()
    if len(returns) == 0:
        return np.nan
    cum = (1 + returns).prod()
    return cum ** (periods_per_year / len(returns)) - 1 if cum > 0 else np.nan


def compute_annualized_volatility(returns, periods_per_year=8760):
    returns = pd.Series(returns).dropna()
    return returns.std(ddof=1) * np.sqrt(periods_per_year) if len(returns) >= 2 else np.nan


def compute_annualized_sharpe(returns, risk_free_rate=0.0, periods_per_year=8760):
    returns = pd.Series(returns).dropna()
    if len(returns) < 2:
        return np.nan
    excess = returns - risk_free_rate / periods_per_year
    vol = excess.std(ddof=1)
    return excess.mean() / vol * np.sqrt(periods_per_year) if vol > 0 else np.nan


def compute_max_drawdown(returns):
    returns = pd.Series(returns).dropna()
    if len(returns) == 0:
        return np.nan
    eq = (1 + returns).cumprod()
    return (eq / eq.cummax() - 1).min()


def compute_turnover(position):
    position = pd.Series(position).fillna(0)
    return position.diff().abs().fillna(position.iloc[0]).sum()


def compute_num_trades(position):
    position = pd.Series(position).fillna(0)
    return int((position.diff().abs().fillna(position.iloc[0]) > 0).sum())


def compute_signal_returns(backtest_df, return_col='net_return', position_col='position'):
    mask = backtest_df[position_col].fillna(0) != 0
    return pd.Series(backtest_df.loc[mask, return_col]).dropna().reset_index(drop=True)


def compute_avg_signal_return(backtest_df, return_col='net_return', position_col='position'):
    signal_returns = compute_signal_returns(backtest_df, return_col=return_col, position_col=position_col)
    if len(signal_returns) == 0:
        return np.nan
    return float(signal_returns.mean())


def compute_signal_return_std(backtest_df, return_col='net_return', position_col='position'):
    signal_returns = compute_signal_returns(backtest_df, return_col=return_col, position_col=position_col)
    return signal_returns.std(ddof=1) if len(signal_returns) >= 2 else np.nan


def compute_paper_sharpe(backtest_df, return_col='net_return', position_col='position'):
    signal_returns = compute_signal_returns(backtest_df, return_col=return_col, position_col=position_col)
    n_signals = len(signal_returns)
    if n_signals == 0:
        return np.nan
    total_return = float((1.0 + signal_returns).prod() - 1.0)
    avg_signal_return = total_return / n_signals
    signal_std = signal_returns.std(ddof=1) if n_signals >= 2 else np.nan
    if pd.isna(signal_std) or signal_std <= 0:
        return np.nan
    return avg_signal_return / signal_std


def summarize_performance(backtest_df, return_col='net_return', position_col='position', periods_per_year=8760, risk_free_rate=0.0):
    returns = backtest_df[return_col].dropna()
    position = backtest_df[position_col]
    return pd.Series({
        'Cumulative Return': compute_cumulative_return(returns),
        'Annualized Return': compute_annualized_return(returns, periods_per_year),
        'Annualized Volatility': compute_annualized_volatility(returns, periods_per_year),
        'Annualized Sharpe': compute_annualized_sharpe(returns, risk_free_rate, periods_per_year),
        'Maximum Drawdown': compute_max_drawdown(returns),
        'Turnover': compute_turnover(position),
        'Number of Trades': compute_num_trades(position),
        'Avg Return': compute_avg_signal_return(backtest_df, return_col=return_col, position_col=position_col),
        'Signal Return Std': compute_signal_return_std(backtest_df, return_col=return_col, position_col=position_col),
    })

In [ ]:
# -- Directional metrics and backtest helpers -----------
def _compute_long_metrics(position, raw_return):
    pred = position > 0
    actual = raw_return > 0
    tp = int((pred & actual).sum())
    pp = int(pred.sum())
    ap = int(actual.sum())
    precision = tp / pp if pp > 0 else np.nan
    recall = tp / ap if ap > 0 else np.nan
    return precision, recall


def _compute_short_metrics(position, raw_return):
    pred = position < 0
    actual = raw_return < 0
    tp = int((pred & actual).sum())
    pp = int(pred.sum())
    ap = int(actual.sum())
    precision = tp / pp if pp > 0 else np.nan
    recall = tp / ap if ap > 0 else np.nan
    return precision, recall


def _safe_mean_or_nan(values):
    valid = [v for v in values if pd.notna(v)]
    return float(np.mean(valid)) if valid else np.nan


def compute_directional_metrics(backtest_df, mode='long_only'):
    position = backtest_df['position'].fillna(0).values
    raw_return = backtest_df['raw_return'].fillna(0).values
    if mode == 'long_only':
        precision, recall = _compute_long_metrics(position, raw_return)
    elif mode == 'short_only':
        precision, recall = _compute_short_metrics(position, raw_return)
    elif mode == 'long_short':
        p_long, r_long = _compute_long_metrics(position, raw_return)
        p_short, r_short = _compute_short_metrics(position, raw_return)
        precision = _safe_mean_or_nan([p_long, p_short])
        recall = _safe_mean_or_nan([r_long, r_short])
    else:
        raise ValueError(f'Unsupported mode: {mode}')
    return pd.Series({'Precision': precision, 'Recall': recall})


def _sigmoid_score(score, scale=20.0):
    score = np.asarray(score, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(scale * score, -50, 50)))


def build_positions_from_dual_probabilities(long_prob, short_prob, mode='long_only', long_threshold=0.5, short_threshold=0.5):
    long_prob = np.asarray(long_prob, dtype=float)
    short_prob = np.asarray(short_prob if short_prob is not None else np.zeros_like(long_prob), dtype=float)
    long_threshold = 0.5 if long_threshold is None or pd.isna(long_threshold) else float(long_threshold)
    short_threshold = 0.5 if short_threshold is None or pd.isna(short_threshold) else float(short_threshold)

    if mode == 'long_only':
        return (long_prob >= long_threshold).astype(int)

    if mode == 'short_only':
        pos = np.zeros(len(short_prob), dtype=int)
        pos[short_prob >= short_threshold] = -1
        return pos

    if mode == 'long_short':
        pos = np.zeros(len(long_prob), dtype=int)
        long_active = long_prob >= long_threshold
        short_active = short_prob >= short_threshold
        long_margin = long_prob - long_threshold
        short_margin = short_prob - short_threshold

        pos[long_active & ~short_active] = 1
        pos[short_active & ~long_active] = -1

        both = long_active & short_active
        pos[both & (long_margin > short_margin)] = 1
        pos[both & (short_margin > long_margin)] = -1
        return pos

    raise ValueError(f'Unsupported mode: {mode}')


def run_backtest_from_position(df, position, return_col=RETURN_COL, cost=TRANSACTION_COST):
    out = df.copy().reset_index(drop=True)
    out['position'] = pd.Series(position).astype(float)
    out['raw_return'] = out[return_col].astype(float)
    if LOG_RETURN_COL in out.columns:
        out['raw_log_return'] = out[LOG_RETURN_COL].astype(float)
    out['position_change'] = (out['position'] - out['position'].shift(1).fillna(0)).abs()
    out['transaction_cost'] = cost * out['position_change']
    out['gross_return'] = out['position'] * out['raw_return']
    out['net_return'] = out['gross_return'] - out['transaction_cost']
    out['gross_equity_curve'] = (1 + out['gross_return']).cumprod()
    out['net_equity_curve'] = (1 + out['net_return']).cumprod()
    return out


def summarize_signal_performance(backtest_df, mode='long_only'):
    perf = summarize_performance(backtest_df)
    dirm = compute_directional_metrics(backtest_df, mode=mode)
    return {
        'Sharpe': compute_paper_sharpe(backtest_df),
        'Precision': dirm['Precision'],
        'Recall': dirm['Recall'],
        'Avg Return': perf['Avg Return'],
        'Cumulative Return': perf['Cumulative Return'],
        'Turnover': perf['Turnover'],
        'Number of Trades': perf['Number of Trades'],
        'Maximum Drawdown': perf['Maximum Drawdown'],
        'Annualized Sharpe': perf['Annualized Sharpe'],
    }

In [ ]:
# -- Threshold tuning -----------------------------------
def _threshold_score(backtest_df, mode='long_only'):
    metrics = summarize_signal_performance(backtest_df, mode=mode)
    trades = int(metrics['Number of Trades'])
    signal_ratio = float((backtest_df['position'] != 0).mean())
    if trades < MIN_TRADES_VALID or signal_ratio < MIN_SIGNAL_RATIO:
        return -np.inf, {**metrics, 'Trades': trades, 'Signal Ratio': signal_ratio}
    if pd.isna(metrics['Recall']) or metrics['Recall'] < MIN_PRECISION_RECALL:
        return -np.inf, {**metrics, 'Trades': trades, 'Signal Ratio': signal_ratio}
    score = metrics['Precision'] if pd.notna(metrics['Precision']) else -np.inf
    return score, {**metrics, 'Trades': trades, 'Signal Ratio': signal_ratio}


def tune_threshold_precision(long_prob, short_prob, val_df, mode='long_only', long_grid=None, short_grid=None, cost=0.00035):
    long_grid = list(long_grid if long_grid is not None else LONG_T_GRID)
    short_grid = list(short_grid if short_grid is not None else SHORT_T_GRID)
    fallback = {
        'long_threshold': long_grid[len(long_grid) // 2] if len(long_grid) > 0 else 0.5,
        'short_threshold': short_grid[len(short_grid) // 2] if len(short_grid) > 0 else 0.5,
        'Validation Precision': np.nan,
        'Validation Recall': np.nan,
        'Validation Sharpe': np.nan,
        'Validation Avg Return': np.nan,
        'Validation Trades': np.nan,
        'Validation Signal Ratio': np.nan,
        'Threshold Feasible': False,
    }
    if len(val_df) == 0:
        return fallback

    best_key = None
    best = None

    if mode == 'long_only':
        for lt in long_grid:
            pos = build_positions_from_dual_probabilities(long_prob, short_prob, mode='long_only', long_threshold=lt)
            bt = run_backtest_from_position(val_df, pos, return_col=RETURN_COL, cost=cost)
            score, diag = _threshold_score(bt, mode='long_only')
            if not np.isfinite(score):
                continue
            key = (
                score,
                diag['Recall'] if pd.notna(diag['Recall']) else -np.inf,
                diag['Sharpe'] if pd.notna(diag['Sharpe']) else -np.inf,
            )
            if best_key is None or key > best_key:
                best_key = key
                best = {
                    'long_threshold': lt,
                    'short_threshold': None,
                    'Validation Precision': diag['Precision'],
                    'Validation Recall': diag['Recall'],
                    'Validation Sharpe': diag['Sharpe'],
                    'Validation Avg Return': diag['Avg Return'],
                    'Validation Trades': diag['Trades'],
                    'Validation Signal Ratio': diag['Signal Ratio'],
                    'Threshold Feasible': True,
                }
        return best if best is not None else fallback

    if mode == 'short_only':
        for st in short_grid:
            pos = build_positions_from_dual_probabilities(long_prob, short_prob, mode='short_only', short_threshold=st)
            bt = run_backtest_from_position(val_df, pos, return_col=RETURN_COL, cost=cost)
            score, diag = _threshold_score(bt, mode='short_only')
            if not np.isfinite(score):
                continue
            key = (
                score,
                diag['Recall'] if pd.notna(diag['Recall']) else -np.inf,
                diag['Sharpe'] if pd.notna(diag['Sharpe']) else -np.inf,
            )
            if best_key is None or key > best_key:
                best_key = key
                best = {
                    'long_threshold': None,
                    'short_threshold': st,
                    'Validation Precision': diag['Precision'],
                    'Validation Recall': diag['Recall'],
                    'Validation Sharpe': diag['Sharpe'],
                    'Validation Avg Return': diag['Avg Return'],
                    'Validation Trades': diag['Trades'],
                    'Validation Signal Ratio': diag['Signal Ratio'],
                    'Threshold Feasible': True,
                }
        return best if best is not None else fallback

    if mode == 'long_short':
        for lt in long_grid:
            for st in short_grid:
                pos = build_positions_from_dual_probabilities(long_prob, short_prob, mode='long_short', long_threshold=lt, short_threshold=st)
                bt = run_backtest_from_position(val_df, pos, return_col=RETURN_COL, cost=cost)
                score, diag = _threshold_score(bt, mode='long_short')
                if not np.isfinite(score):
                    continue
                key = (
                    score,
                    diag['Recall'] if pd.notna(diag['Recall']) else -np.inf,
                    diag['Sharpe'] if pd.notna(diag['Sharpe']) else -np.inf,
                )
                if best_key is None or key > best_key:
                    best_key = key
                    best = {
                        'long_threshold': lt,
                        'short_threshold': st,
                        'Validation Precision': diag['Precision'],
                        'Validation Recall': diag['Recall'],
                        'Validation Sharpe': diag['Sharpe'],
                        'Validation Avg Return': diag['Avg Return'],
                        'Validation Trades': diag['Trades'],
                        'Validation Signal Ratio': diag['Signal Ratio'],
                        'Threshold Feasible': True,
                    }
        return best if best is not None else fallback

    raise ValueError(f'Unsupported mode: {mode}')

In [ ]:
# -- Bootstrap helpers ----------------------------------
def paper_block_length(n):
    return max(1, int(np.floor(n ** (1 / 3))))


def block_bootstrap_mean_diff(a, b, block=None, n_boot=10000, seed=GLOBAL_SEED, alternative='two-sided'):
    a = np.asarray(pd.Series(a).dropna(), dtype=float)
    b = np.asarray(pd.Series(b).dropna(), dtype=float)
    n = min(len(a), len(b))
    if n < 4:
        return {'mean_diff': np.nan, 'p_value': np.nan, 'block_length': np.nan, 'n_boot': n_boot}
    block = paper_block_length(n) if block is None else int(block)
    n_blocks = n // block
    if n_blocks < 2:
        return {'mean_diff': np.nan, 'p_value': np.nan, 'block_length': block, 'n_boot': n_boot}

    n_trim = n_blocks * block
    a_blocks = a[:n_trim].reshape(n_blocks, block)
    b_blocks = b[:n_trim].reshape(n_blocks, block)
    obs = float(a_blocks.reshape(-1).mean() - b_blocks.reshape(-1).mean())
    a_null = a_blocks - a_blocks.mean()
    b_null = b_blocks - b_blocks.mean()
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n_blocks, size=n_blocks)
        boot[i] = float(a_null[idx].reshape(-1).mean() - b_null[idx].reshape(-1).mean())

    if alternative == 'greater':
        p_value = float(np.mean(boot >= obs))
    elif alternative == 'less':
        p_value = float(np.mean(boot <= obs))
    else:
        p_value = float(np.mean(np.abs(boot) >= abs(obs)))
    return {'mean_diff': obs, 'p_value': p_value, 'block_length': block, 'n_boot': n_boot}


def block_bootstrap_mean_vs_zero(a, block=None, n_boot=10000, seed=GLOBAL_SEED, alternative='greater'):
    a = np.asarray(pd.Series(a).dropna(), dtype=float)
    n = len(a)
    if n < 4:
        return {'mean': np.nan, 'p_value': np.nan, 'block_length': np.nan, 'n_boot': n_boot}
    block = paper_block_length(n) if block is None else int(block)
    n_blocks = n // block
    if n_blocks < 2:
        return {'mean': np.nan, 'p_value': np.nan, 'block_length': block, 'n_boot': n_boot}

    n_trim = n_blocks * block
    a_blocks = a[:n_trim].reshape(n_blocks, block)
    obs = float(a_blocks.reshape(-1).mean())
    a_null = a_blocks - a_blocks.mean()
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n_blocks, size=n_blocks)
        boot[i] = float(a_null[idx].reshape(-1).mean())

    if alternative == 'greater':
        p_value = float(np.mean(boot >= obs))
    elif alternative == 'less':
        p_value = float(np.mean(boot <= obs))
    else:
        p_value = float(np.mean(np.abs(boot) >= abs(obs)))
    return {'mean': obs, 'p_value': p_value, 'block_length': block, 'n_boot': n_boot}

## 5. Buy-and-Hold Baseline
Buy-and-hold is exported separately for every asset because it serves two roles in the report:
- the passive benchmark in the equity-curve plots,
- the paired benchmark in the usefulness tests.


In [14]:
def backtest_buy_and_hold(test_df, return_col=RETURN_COL, transaction_cost=TRANSACTION_COST):
    df = test_df.dropna(subset=[return_col]).copy().reset_index(drop=True)
    df['position'] = 1.0
    df['raw_return'] = df[return_col].astype(float)
    if LOG_RETURN_COL in df.columns:
        df['raw_log_return'] = df[LOG_RETURN_COL].astype(float)
    df['position_change'] = (df['position'] - df['position'].shift(1).fillna(0)).abs()
    df['transaction_cost'] = transaction_cost * df['position_change']
    df['gross_return'] = df['position'] * df['raw_return']
    df['net_return'] = df['gross_return'] - df['transaction_cost']
    df['gross_equity_curve'] = (1 + df['gross_return']).cumprod()
    df['net_equity_curve'] = (1 + df['net_return']).cumprod()
    return df


In [ ]:
bh_summaries = []

for asset in ASSETS:
    bt = backtest_buy_and_hold(test_data[asset])
    perf = summarize_performance(bt)
    dirm = compute_directional_metrics(bt)
    row = pd.concat([perf, dirm]).to_frame().T
    row.insert(0, 'Asset', asset)
    row.insert(1, 'Strategy', 'Buy-and-Hold')
    bh_summaries.append(row)
    bt.to_csv(RESULT_DIR / 'buy_and_hold' / f'{asset.lower()}_backtest.csv', index=False)

bh_summary = pd.concat(bh_summaries, ignore_index=True)
bh_summary.to_csv(RESULT_DIR / 'buy_and_hold' / 'summary_all_assets.csv', index=False)
print('Buy-and-Hold results:')
display(bh_summary[['Asset', 'Cumulative Return', 'Annualized Sharpe', 'Maximum Drawdown', 'Number of Trades']])

Buy-and-Hold results:


,Asset,Cumulative Return,Annualized Sharpe,Maximum Drawdown,Number of Trades
0,BTC,2.299935,2.147217,-0.323107,1.0
1,ETH,0.930191,1.167748,-0.466551,1.0
2,XRP,2.967493,1.715271,-0.457828,1.0
3,SOL,2.342186,1.635658,-0.465184,1.0


## 6. Neural Sequence Helpers Used by the Main Grid
Only the shared sequence-building utilities remain here.

The standalone model-specific benchmark branches were removed, so the unified experiment grid is the only modeling path that now needs maintenance.


In [ ]:
def make_sequence_dataset(df, features, target_col=TARGET_LONG_COL, return_col=RETURN_COL, lookback=24):
    d = df.dropna(subset=features + [target_col, return_col]).reset_index(drop=True).copy()
    X_raw = d[features].values
    y_raw = d[target_col].values.astype(int)
    r_raw = d[return_col].values.astype(float)
    X_seq, y_seq, r_seq = [], [], []
    for i in range(lookback - 1, len(d)):
        X_seq.append(X_raw[i - lookback + 1:i + 1])
        y_seq.append(y_raw[i])
        r_seq.append(r_raw[i])
    return np.array(X_seq), np.array(y_seq), np.array(r_seq), d.iloc[lookback - 1:].reset_index(drop=True)

In [ ]:
def build_gru_model(n_steps, n_features):
    # The raw-feature GRU can collapse into near-constant probabilities if regularization is too aggressive.
    # A slightly larger recurrent layer, default GRU activations, and lighter dropout keep the sequence model expressive
    # enough to emit usable probability dispersion for threshold tuning.
    model = keras.Sequential([
        layers.Input(shape=(n_steps, n_features)),
        layers.GRU(16, dropout=0.2, recurrent_dropout=0.0),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

## 7. Main Experiment Grid and Standardized Trading Logs
This is the core experimental section of the notebook.

The section is split into smaller subsections so the training, evaluation, and orchestration paths are easier to read and maintain.

For each asset, feature set, model family, strategy mode, and cost regime, the pipeline:
1. fits separate long and short classifiers for ML models,
2. tunes validation thresholds by precision with guardrails,
3. builds `{0,1}`, `{0,-1}`, or `{0,1,-1}` positions depending on the strategy mode,
4. runs the frozen-threshold test backtest,
5. writes one standardized trading log per configuration,
6. stores one summary row for downstream reporting.


### 7.1 Shared Metadata and Dataset Caches
Shared feature metadata, reusable dataset bundles, and cache-backed base frames used across the experiment runners.


In [ ]:
# -- Main-grid shared metadata, caches, and artifact helpers --
if RUN_UNIFIED_GRID:
    TABULAR_BUNDLE_CACHE = {}
    RULE_BASE_FRAME_CACHE = {}
    RULE_SIGNAL_CACHE = {}


    def _model_artifact_dir(asset, model_name, feature_set_name):
        model_slug = model_name.lower().replace(' ', '_')
        out_dir = MODELS_DIR / model_slug / feature_set_name / asset.lower()
        out_dir.mkdir(parents=True, exist_ok=True)
        return out_dir


    def _write_model_metadata(out_dir, payload):
        (out_dir / 'metadata.json').write_text(json.dumps(payload, indent=2, sort_keys=True), encoding='utf-8')


    def _save_tabular_model_bundle(asset, model_name, feature_set_name, features, fit_result):
        out_dir = _model_artifact_dir(asset, model_name, feature_set_name)
        artifacts = ['long_model.joblib', 'short_model.joblib']
        joblib.dump(fit_result['long_model'], out_dir / 'long_model.joblib')
        joblib.dump(fit_result['short_model'], out_dir / 'short_model.joblib')
        if fit_result.get('scaler') is not None:
            joblib.dump(fit_result['scaler'], out_dir / 'scaler.joblib')
            artifacts.append('scaler.joblib')
        artifacts.append('metadata.json')
        _write_model_metadata(out_dir, {
            'asset': asset,
            'model': model_name,
            'feature_set': feature_set_name,
            'features': list(features),
            'artifacts': artifacts,
        })
        return out_dir


    def _save_sequence_model_bundle(asset, model_name, feature_set_name, features, long_obj, short_obj):
        out_dir = _model_artifact_dir(asset, model_name, feature_set_name)
        long_obj['model'].save(out_dir / 'long_model.keras', include_optimizer=False)
        short_obj['model'].save(out_dir / 'short_model.keras', include_optimizer=False)
        joblib.dump(long_obj['scaler'], out_dir / 'long_scaler.joblib')
        joblib.dump(short_obj['scaler'], out_dir / 'short_scaler.joblib')
        _write_model_metadata(out_dir, {
            'asset': asset,
            'model': model_name,
            'feature_set': feature_set_name,
            'features': list(features),
            'lookback_hours': int(long_obj['lookback']),
            'artifacts': ['long_model.keras', 'short_model.keras', 'long_scaler.joblib', 'short_scaler.joblib', 'metadata.json'],
        })
        return out_dir


    def _feature_metadata(feature_set_name):
        if feature_set_name == 'rule_based':
            return {'Input Type': 'rule', 'Feature Type': 'rule'}
        return {
            'Input Type': 'candle' if feature_set_name.startswith('candle') else 'ohlc',
            'Feature Type': 'extended' if feature_set_name.endswith('extended') else 'raw',
        }


    def _get_usable_features(asset, feature_set_name):
        feats = FEATURE_SET_DEFS[feature_set_name]
        tr = train_data[asset]
        vl = val_data[asset]
        te = test_data[asset]
        return [c for c in feats if c in tr.columns and c in vl.columns and c in te.columns]


    def _prepare_tabular_bundle(asset, feature_set_name):
        key = (asset, feature_set_name)
        if key in TABULAR_BUNDLE_CACHE:
            return TABULAR_BUNDLE_CACHE[key]

        features = _get_usable_features(asset, feature_set_name)
        if len(features) == 0:
            TABULAR_BUNDLE_CACHE[key] = None
            return None

        required = features + [TARGET_LONG_COL, TARGET_SHORT_COL, RETURN_COL, LOG_RETURN_COL]
        tr = train_data[asset].dropna(subset=required).copy().reset_index(drop=True)
        vl = val_data[asset].dropna(subset=required).copy().reset_index(drop=True)
        te = test_data[asset].dropna(subset=required).copy().reset_index(drop=True)
        if len(tr) == 0 or len(vl) == 0 or len(te) == 0:
            TABULAR_BUNDLE_CACHE[key] = None
            return None

        ref_cols = ['datetime', RETURN_COL, LOG_RETURN_COL]
        bundle = {
            'asset': asset,
            'feature_set': feature_set_name,
            'features': features,
            'X_tr': tr[features].to_numpy(dtype=np.float32, copy=True),
            'X_vl': vl[features].to_numpy(dtype=np.float32, copy=True),
            'X_te': te[features].to_numpy(dtype=np.float32, copy=True),
            'y_tr_long': tr[TARGET_LONG_COL].to_numpy(dtype=np.int8, copy=True),
            'y_tr_short': tr[TARGET_SHORT_COL].to_numpy(dtype=np.int8, copy=True),
            'vl_ref': vl[ref_cols].copy().reset_index(drop=True),
            'te_ref': te[ref_cols].copy().reset_index(drop=True),
        }
        TABULAR_BUNDLE_CACHE[key] = bundle
        return bundle


    def _get_rule_base_frame(asset, split_name):
        key = (asset, split_name)
        if key in RULE_BASE_FRAME_CACHE:
            return RULE_BASE_FRAME_CACHE[key]
        source = val_data if split_name == 'val' else test_data
        frame = source[asset].dropna(subset=['close', RETURN_COL, LOG_RETURN_COL]).copy().reset_index(drop=True)
        RULE_BASE_FRAME_CACHE[key] = frame
        return frame

### 7.2 Shared Evaluation and Logging Contract
Common threshold tuning, standardized trading-log export, and summary-row construction shared by all experiment families.


In [ ]:
# -- Main-grid evaluation, logging, and row-construction helpers --
if RUN_UNIFIED_GRID:
    def _standardize_trading_log(bt, asset, model_name, feature_set, strategy_mode, cost_label, long_prob, short_prob, thresholds, model_params=None):
        df = bt.copy().reset_index(drop=True)
        if 'datetime' not in df.columns:
            df['datetime'] = pd.NaT
        prev_pos = df['position'].shift(1).fillna(0)

        def _action(p, pp):
            p, pp = int(p), int(pp)
            if p == pp:
                return 'HOLD'
            if pp == 0 and p == 1:
                return 'OPEN_LONG'
            if pp == 0 and p == -1:
                return 'OPEN_SHORT'
            if pp == 1 and p == 0:
                return 'CLOSE_LONG'
            if pp == -1 and p == 0:
                return 'CLOSE_SHORT'
            if pp == 1 and p == -1:
                return 'FLIP_LONG_TO_SHORT'
            if pp == -1 and p == 1:
                return 'FLIP_SHORT_TO_LONG'
            return 'CHANGE'

        df['trade_action'] = [_action(p, pp) for p, pp in zip(df['position'].fillna(0), prev_pos)]
        df['signal'] = np.where(df['position'] > 0, 'LONG', np.where(df['position'] < 0, 'SHORT', 'FLAT'))
        df['is_trade_event'] = (df['position_change'] > 0).astype(int)
        df['asset'] = asset
        df['model'] = model_name
        df['feature_set'] = feature_set
        df['strategy_mode'] = strategy_mode
        df['cost_regime'] = cost_label
        df['y_prob_long'] = np.asarray(long_prob, dtype=float)[:len(df)]
        df['y_prob_short'] = np.asarray(short_prob, dtype=float)[:len(df)] if short_prob is not None else np.nan
        df['long_threshold'] = thresholds.get('long_threshold')
        df['short_threshold'] = thresholds.get('short_threshold')
        df['threshold_feasible'] = thresholds.get('Threshold Feasible', False)
        df['model_params'] = json.dumps(model_params or {}, sort_keys=True)

        out_cols = [
            'datetime', 'asset', 'model', 'feature_set', 'strategy_mode', 'cost_regime',
            'signal', 'trade_action', 'is_trade_event',
            'y_prob_long', 'y_prob_short', 'long_threshold', 'short_threshold', 'threshold_feasible', 'model_params',
            'position', 'position_change', 'transaction_cost',
            'raw_return', 'raw_log_return', 'gross_return', 'net_return', 'gross_equity_curve', 'net_equity_curve',
        ]
        return df[[c for c in out_cols if c in df.columns]].copy()


    def _build_result_row(asset, model_name, feature_set, strategy_mode, cost_label, thresholds, validation_metrics, test_metrics, log_path, model_family='ML', model_params=None):
        meta = _feature_metadata(feature_set)
        return {
            'Asset': asset,
            'Model': model_name,
            'Model Family': model_family,
            'Feature Set': feature_set,
            'Input Type': meta['Input Type'],
            'Feature Type': meta['Feature Type'],
            'Strategy Mode': strategy_mode,
            'Strategy Label': {'long_only': 'long', 'short_only': 'short', 'long_short': 'long_short'}[strategy_mode],
            'Cost Regime': cost_label,
            'Long Threshold': thresholds.get('long_threshold'),
            'Short Threshold': thresholds.get('short_threshold'),
            'Threshold Feasible': thresholds.get('Threshold Feasible', False),
            'Validation Sharpe': validation_metrics['Sharpe'],
            'Validation Precision': validation_metrics['Precision'],
            'Validation Recall': validation_metrics['Recall'],
            'Validation Avg Return': validation_metrics['Avg Return'],
            'Validation Trades': validation_metrics['Trades'],
            'Validation Signal Ratio': validation_metrics['Signal Ratio'],
            'Test Sharpe': test_metrics['Sharpe'],
            'Test Precision': test_metrics['Precision'],
            'Test Recall': test_metrics['Recall'],
            'Test Avg Return': test_metrics['Avg Return'],
            'Cumulative Return': test_metrics['Cumulative Return'],
            'Maximum Drawdown': test_metrics['Maximum Drawdown'],
            'Turnover': test_metrics['Turnover'],
            'Number of Trades': test_metrics['Number of Trades'],
            'Annualized Sharpe': test_metrics['Annualized Sharpe'],
            'Model Params': json.dumps(model_params or {}, sort_keys=True),
            'Log Path': str(log_path),
        }


    def _evaluate_dual_probabilities(asset, model_name, feature_set, long_prob_v, short_prob_v, vl_ref, long_prob_t, short_prob_t, te_ref, model_family='ML', model_params=None, long_grid=None, short_grid=None):
        rows = []
        # Each fitted model produces one exported row per strategy-mode / cost pair.
        for strategy_mode in UNIFIED_STRATEGY_MODES:
            for cost_label, cost_value in COST_REGIMES.items():
                thresholds = tune_threshold_precision(
                    long_prob_v,
                    short_prob_v,
                    vl_ref,
                    mode=strategy_mode,
                    long_grid=long_grid,
                    short_grid=short_grid,
                    cost=cost_value,
                )
                pos_t = build_positions_from_dual_probabilities(
                    long_prob_t,
                    short_prob_t,
                    mode=strategy_mode,
                    long_threshold=thresholds.get('long_threshold'),
                    short_threshold=thresholds.get('short_threshold'),
                )
                bt_t = run_backtest_from_position(te_ref, pos_t, return_col=RETURN_COL, cost=cost_value)
                validation_metrics = {
                    'Sharpe': thresholds.get('Validation Sharpe'),
                    'Precision': thresholds.get('Validation Precision'),
                    'Recall': thresholds.get('Validation Recall'),
                    'Avg Return': thresholds.get('Validation Avg Return'),
                    'Trades': thresholds.get('Validation Trades'),
                    'Signal Ratio': thresholds.get('Validation Signal Ratio'),
                }
                test_metrics = summarize_signal_performance(bt_t, mode=strategy_mode)
                out_dir = DATA_LOG_DIR / cost_label / strategy_mode / model_name.lower().replace(' ', '_') / feature_set / asset.lower()
                out_dir.mkdir(parents=True, exist_ok=True)
                out_path = out_dir / 'trading_log.csv'
                log_df = _standardize_trading_log(
                    bt_t,
                    asset,
                    model_name,
                    feature_set,
                    strategy_mode,
                    cost_label,
                    long_prob_t,
                    short_prob_t,
                    thresholds,
                    model_params=model_params,
                )
                log_df.to_csv(out_path, index=False)
                rows.append(
                    _build_result_row(
                        asset,
                        model_name,
                        feature_set,
                        strategy_mode,
                        cost_label,
                        thresholds,
                        validation_metrics,
                        test_metrics,
                        out_path,
                        model_family=model_family,
                        model_params=model_params,
                    )
                )
        return rows

### 7.3 Tabular ML Grid
Execution helpers for Logistic Regression, XGBoost, and LightGBM across the four paper-aligned feature sets.


In [ ]:
# -- Main-grid tabular-model fitting helpers ------------
if RUN_UNIFIED_GRID:
    def _fit_logistic_pair(bundle):
        # Logistic regression uses one scaler shared across the long and short classifiers.
        scaler = StandardScaler()
        X_tr = np.asarray(bundle['X_tr'], dtype=np.float32)
        X_vl = np.asarray(bundle['X_vl'], dtype=np.float32)
        X_te = np.asarray(bundle['X_te'], dtype=np.float32)
        y_tr_long = np.asarray(bundle['y_tr_long'], dtype=np.int8)
        y_tr_short = np.asarray(bundle['y_tr_short'], dtype=np.int8)
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_vl_scaled = scaler.transform(X_vl)
        X_te_scaled = scaler.transform(X_te)

        debug_print(f"Fitting Logistic Regression long classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
        long_model = LogisticRegression(max_iter=2000, random_state=GLOBAL_SEED, C=1.0, verbose=LOGISTIC_MODEL_VERBOSE)
        debug_print(f"Fitting Logistic Regression short classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]}")
        short_model = LogisticRegression(max_iter=2000, random_state=GLOBAL_SEED, C=1.0, verbose=LOGISTIC_MODEL_VERBOSE)
        long_model.fit(X_tr_scaled, y_tr_long)
        short_model.fit(X_tr_scaled, y_tr_short)

        return {
            'long_prob_v': long_model.predict_proba(X_vl_scaled)[:, 1],
            'short_prob_v': short_model.predict_proba(X_vl_scaled)[:, 1],
            'long_prob_t': long_model.predict_proba(X_te_scaled)[:, 1],
            'short_prob_t': short_model.predict_proba(X_te_scaled)[:, 1],
            'model_params': {},
            'long_model': long_model,
            'short_model': short_model,
            'scaler': scaler,
        }


    def _fit_tree_pair(model_name, bundle):
        # Tree models consume the same cached bundle but do not require feature scaling.
        X_tr = np.asarray(bundle['X_tr'], dtype=np.float32)
        X_vl = np.asarray(bundle['X_vl'], dtype=np.float32)
        X_te = np.asarray(bundle['X_te'], dtype=np.float32)
        y_tr_long = np.asarray(bundle['y_tr_long'], dtype=np.int8)
        y_tr_short = np.asarray(bundle['y_tr_short'], dtype=np.int8)

        if model_name == 'XGBoost':
            X_fit = X_tr
            X_val = X_vl
            X_test = X_te
            params = {
                'n_estimators': 100,
                'eval_metric': 'logloss',
                'random_state': GLOBAL_SEED,
                'n_jobs': TREE_MODEL_N_JOBS,
                'verbosity': XGBOOST_MODEL_VERBOSITY,
            }
        elif model_name == 'LightGBM':
            feature_names = list(bundle['features'])
            X_fit = pd.DataFrame(X_tr, columns=feature_names)
            X_val = pd.DataFrame(X_vl, columns=feature_names)
            X_test = pd.DataFrame(X_te, columns=feature_names)
            params = {
                'n_estimators': 100,
                'random_state': GLOBAL_SEED,
                'n_jobs': TREE_MODEL_N_JOBS,
                'verbosity': LIGHTGBM_MODEL_VERBOSITY,
            }
        else:
            raise ValueError(f'Unsupported tree model: {model_name}')

        debug_print(f"Fitting {model_name} long classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]} | device=cpu")
        long_model = (xgb.XGBClassifier if model_name == 'XGBoost' else lgb.LGBMClassifier)(**params)
        debug_print(f"Fitting {model_name} short classifier | rows={len(bundle['X_tr'])} | features={bundle['X_tr'].shape[1]} | device=cpu")
        short_model = (xgb.XGBClassifier if model_name == 'XGBoost' else lgb.LGBMClassifier)(**params)
        long_model.fit(X_fit, y_tr_long)
        short_model.fit(X_fit, y_tr_short)

        return {
            'long_prob_v': long_model.predict_proba(X_val)[:, 1],
            'short_prob_v': short_model.predict_proba(X_val)[:, 1],
            'long_prob_t': long_model.predict_proba(X_test)[:, 1],
            'short_prob_t': short_model.predict_proba(X_test)[:, 1],
            'model_params': {'tree_device': 'cpu'},
            'long_model': long_model,
            'short_model': short_model,
            'scaler': None,
        }


    def _run_single_tabular_ml_task(asset, feature_set, model_name):
        debug_print(f'Tabular task start | asset={asset} | feature_set={feature_set} | model={model_name}')
        bundle = _prepare_tabular_bundle(asset, feature_set)
        if bundle is None:
            debug_print(f'Tabular task skipped | asset={asset} | feature_set={feature_set} | model={model_name}')
            return []

        if model_name == 'Logistic Regression':
            probs = _fit_logistic_pair(bundle)
        else:
            probs = _fit_tree_pair(model_name, bundle)

        model_dir = _save_tabular_model_bundle(asset, model_name, feature_set, bundle['features'], probs)
        probs['model_params'] = {**probs['model_params'], 'model_dir': str(model_dir)}

        rows = _evaluate_dual_probabilities(
            asset,
            model_name,
            feature_set,
            probs['long_prob_v'],
            probs['short_prob_v'],
            bundle['vl_ref'],
            probs['long_prob_t'],
            probs['short_prob_t'],
            bundle['te_ref'],
            model_family='ML',
            model_params=probs['model_params'],
        )
        debug_print(f'Tabular task done | asset={asset} | feature_set={feature_set} | model={model_name} | rows={len(rows)}')
        return rows

### 7.4 Sequence ML Grid
Training and inference helpers for the sequence-model branch, keeping its preprocessing and fit loop isolated from the tabular path.


In [ ]:
# -- Main-grid sequence-model training and inference helpers --
if RUN_UNIFIED_GRID:
    def _fit_sequence_classifier(model_name, asset, features, target_col, lookback=LOOKBACK_HOURS):
        tr = train_data[asset].dropna(subset=features + [target_col]).copy()
        vl = val_data[asset].dropna(subset=features + [target_col, RETURN_COL, LOG_RETURN_COL]).copy()
        if len(tr) == 0 or len(vl) == 0:
            return None

        # Sequence models are scaled split-by-split using train-fit statistics only.
        scaler = StandardScaler()
        tr_scaled = tr.copy()
        vl_scaled = vl.copy()
        tr_scaled[features] = scaler.fit_transform(tr[features])
        vl_scaled[features] = scaler.transform(vl[features])

        X_tr, y_tr, _, _ = make_sequence_dataset(tr_scaled, features, target_col=target_col, return_col=RETURN_COL, lookback=lookback)
        X_vl, y_vl, _, vl_ref = make_sequence_dataset(vl_scaled, features, target_col=target_col, return_col=RETURN_COL, lookback=lookback)
        if len(X_tr) == 0 or len(X_vl) == 0:
            return None

        mdl = build_gru_model(lookback, len(features))
        debug_print(f'Fitting {model_name} long/short sequence classifier target={target_col} | train_sequences={len(X_tr)} | val_sequences={len(X_vl)} | features={len(features)}')
        es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=GRU_EARLY_STOPPING_PATIENCE, restore_best_weights=True)
        mdl.fit(X_tr, y_tr, validation_data=(X_vl, y_vl), epochs=500, batch_size=GRU_BATCH_SIZE, verbose=SEQUENCE_MODEL_FIT_VERBOSE, callbacks=[es])
        y_prob_v = mdl.predict(X_vl, verbose=0).ravel()
        n = min(len(y_prob_v), len(vl_ref))
        return {
            'model': mdl,
            'scaler': scaler,
            'lookback': lookback,
            'y_prob_v': y_prob_v[:n],
            'vl_ref': vl_ref.iloc[:n].reset_index(drop=True),
        }


    def _predict_sequence_classifier_test(model_obj, asset, features, target_col):
        te = test_data[asset].dropna(subset=features + [target_col, RETURN_COL, LOG_RETURN_COL]).copy().reset_index(drop=True)
        te_scaled = te.copy()
        te_scaled[features] = model_obj['scaler'].transform(te[features])
        X_te, _, _, te_ref = make_sequence_dataset(te_scaled, features, target_col=target_col, return_col=RETURN_COL, lookback=model_obj['lookback'])
        if len(X_te) == 0:
            return np.array([]), te_ref.iloc[:0].copy()
        y_prob_t = model_obj['model'].predict(X_te, verbose=0).ravel()
        n = min(len(y_prob_t), len(te_ref))
        return y_prob_t[:n], te_ref.iloc[:n].reset_index(drop=True)


    def _run_single_sequence_ml_task(asset, feature_set, model_name):
        debug_print(f'Sequence task start | asset={asset} | feature_set={feature_set} | model={model_name}')
        features = _get_usable_features(asset, feature_set)
        if len(features) == 0:
            return []

        long_obj = _fit_sequence_classifier(model_name, asset, features, TARGET_LONG_COL)
        short_obj = _fit_sequence_classifier(model_name, asset, features, TARGET_SHORT_COL)
        if long_obj is None or short_obj is None:
            debug_print(f'Sequence task skipped | asset={asset} | feature_set={feature_set} | model={model_name}')
            return []

        # Align the long and short branches before threshold tuning so each row refers to the same timestamp.
        n_val = min(len(long_obj['y_prob_v']), len(short_obj['y_prob_v']), len(long_obj['vl_ref']), len(short_obj['vl_ref']))
        long_prob_v = long_obj['y_prob_v'][:n_val]
        short_prob_v = short_obj['y_prob_v'][:n_val]
        vl_ref = long_obj['vl_ref'].iloc[:n_val].reset_index(drop=True)

        long_prob_t, te_ref_long = _predict_sequence_classifier_test(long_obj, asset, features, TARGET_LONG_COL)
        short_prob_t, te_ref_short = _predict_sequence_classifier_test(short_obj, asset, features, TARGET_SHORT_COL)
        n_test = min(len(long_prob_t), len(short_prob_t), len(te_ref_long), len(te_ref_short))
        long_prob_t = long_prob_t[:n_test]
        short_prob_t = short_prob_t[:n_test]
        te_ref = te_ref_long.iloc[:n_test].reset_index(drop=True)

        model_dir = _save_sequence_model_bundle(asset, model_name, feature_set, features, long_obj, short_obj)

        rows = _evaluate_dual_probabilities(
            asset,
            model_name,
            feature_set,
            long_prob_v,
            short_prob_v,
            vl_ref,
            long_prob_t,
            short_prob_t,
            te_ref,
            model_family='ML',
            model_params={'lookback_hours': LOOKBACK_HOURS, 'model_dir': str(model_dir)},
            long_grid=GRU_LONG_T_GRID,
            short_grid=GRU_SHORT_T_GRID,
        )
        keras.backend.clear_session()
        gc.collect()
        debug_print(f'Sequence task done | asset={asset} | feature_set={feature_set} | model={model_name} | rows={len(rows)}')
        return rows

### 7.5 Rule-Based Grid
Signal generation, parameter search, and cached execution for the SMA and RSI benchmark strategies.


In [ ]:
# -- Main-grid rule-based signal helpers and cache access --
if RUN_UNIFIED_GRID:
    def _rule_signal_dataframe(df, model_name, params):
        work = df[['datetime', RETURN_COL, LOG_RETURN_COL]].copy().reset_index(drop=True)
        price = df['close'].astype(float).reset_index(drop=True)

        if model_name == 'SMA':
            sma_short = price.rolling(params['short_window'], min_periods=params['short_window']).mean()
            sma_long = price.rolling(params['long_window'], min_periods=params['long_window']).mean()
            score = sma_short / (sma_long + EPS) - 1
        elif model_name == 'RSI':
            rsi = compute_rsi(price, params['window'])
            score = (50 - rsi) / 50
        else:
            raise ValueError(f'Unsupported rule model: {model_name}')

        # Rule scores are mapped into pseudo-probabilities so they can reuse the shared thresholding path.
        work['long_prob'] = _sigmoid_score(score.fillna(0.0), scale=20.0)
        work['short_prob'] = _sigmoid_score((-score).fillna(0.0), scale=20.0)
        return work.loc[score.notna()].reset_index(drop=True)


    def _rule_param_grid(model_name):
        if model_name == 'SMA':
            return [
                {'short_window': sw, 'long_window': lw}
                for sw in SMA_SHORT_WINDOWS for lw in SMA_LONG_WINDOWS if sw < lw
            ]
        if model_name == 'RSI':
            return [{'window': w} for w in RSI_WINDOWS]
        raise ValueError(f'Unsupported rule model: {model_name}')


    def _rule_signal_key(asset, split_name, model_name, params):
        return (asset, split_name, model_name, tuple(sorted(params.items())))


    def _get_rule_signal_frame(asset, split_name, model_name, params):
        key = _rule_signal_key(asset, split_name, model_name, params)
        if key in RULE_SIGNAL_CACHE:
            return RULE_SIGNAL_CACHE[key]
        signal_df = _rule_signal_dataframe(_get_rule_base_frame(asset, split_name), model_name, params)
        RULE_SIGNAL_CACHE[key] = signal_df
        return signal_df



    def _run_single_rule_task(asset, model_name, strategy_mode, cost_label):
        debug_print(f'Rule task start | asset={asset} | model={model_name} | mode={strategy_mode} | cost={cost_label}')
        cost_value = COST_REGIMES[cost_label]
        best_key = None
        best_payload = None

        # Search the rule parameter grid on validation data before freezing one configuration for test evaluation.
        for params in _rule_param_grid(model_name):
            vl_sig = _get_rule_signal_frame(asset, 'val', model_name, params)
            te_sig = _get_rule_signal_frame(asset, 'test', model_name, params)
            if len(vl_sig) == 0 or len(te_sig) == 0:
                continue

            thresholds = tune_threshold_precision(
                vl_sig['long_prob'].values,
                vl_sig['short_prob'].values,
                vl_sig[['datetime', RETURN_COL, LOG_RETURN_COL]].copy(),
                mode=strategy_mode,
                cost=cost_value,
            )
            if not thresholds.get('Threshold Feasible', False):
                continue

            key = (
                thresholds.get('Validation Precision') if pd.notna(thresholds.get('Validation Precision')) else -np.inf,
                thresholds.get('Validation Recall') if pd.notna(thresholds.get('Validation Recall')) else -np.inf,
                thresholds.get('Validation Sharpe') if pd.notna(thresholds.get('Validation Sharpe')) else -np.inf,
            )
            if best_key is None or key > best_key:
                best_key = key
                best_payload = (params, thresholds, vl_sig, te_sig)

        if best_payload is None:
            debug_print(f'Rule task skipped | asset={asset} | model={model_name} | mode={strategy_mode} | cost={cost_label}')
            return []

        params, thresholds, _, te_sig = best_payload
        pos_t = build_positions_from_dual_probabilities(
            te_sig['long_prob'].values,
            te_sig['short_prob'].values,
            mode=strategy_mode,
            long_threshold=thresholds.get('long_threshold'),
            short_threshold=thresholds.get('short_threshold'),
        )
        te_ref = te_sig[['datetime', RETURN_COL, LOG_RETURN_COL]].copy().reset_index(drop=True)
        bt_t = run_backtest_from_position(te_ref, pos_t, return_col=RETURN_COL, cost=cost_value)
        validation_metrics = {
            'Sharpe': thresholds.get('Validation Sharpe'),
            'Precision': thresholds.get('Validation Precision'),
            'Recall': thresholds.get('Validation Recall'),
            'Avg Return': thresholds.get('Validation Avg Return'),
            'Trades': thresholds.get('Validation Trades'),
            'Signal Ratio': thresholds.get('Validation Signal Ratio'),
        }
        test_metrics = summarize_signal_performance(bt_t, mode=strategy_mode)

        rows = []
        feature_set = 'rule_based'
        out_dir = DATA_LOG_DIR / cost_label / strategy_mode / model_name.lower() / feature_set / asset.lower()
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / 'trading_log.csv'
        log_df = _standardize_trading_log(
            bt_t,
            asset,
            model_name,
            feature_set,
            strategy_mode,
            cost_label,
            te_sig['long_prob'].values,
            te_sig['short_prob'].values,
            thresholds,
            model_params=params,
        )
        log_df.to_csv(out_path, index=False)
        rows.append(
            _build_result_row(
                asset,
                model_name,
                feature_set,
                strategy_mode,
                cost_label,
                thresholds,
                validation_metrics,
                test_metrics,
                out_path,
                model_family='Rule',
                model_params=params,
            )
        )
        debug_print(f'Rule task done | asset={asset} | model={model_name} | mode={strategy_mode} | cost={cost_label} | rows={len(rows)}')
        return rows

### 7.6 Unified Orchestrator
Thin orchestration layer that warms caches, runs each experiment family, concatenates result rows, and writes the unified summary artifact used by downstream reporting sections.


In [23]:
# -- Main-grid orchestration and summary export ----------
if RUN_UNIFIED_GRID:
    debug_print('Warming tabular bundle cache...')
    for asset in ASSETS:
        for feature_set in UNIFIED_FEATURE_SETS:
            _prepare_tabular_bundle(asset, feature_set)


    rows = []

    # Run the experiment families sequentially so artifacts and logs are easier to trace.
    for model_name in TABULAR_ML_MODELS:
        debug_print(f'Running tabular model sequentially | model={model_name}')
        for asset in ASSETS:
            for feature_set in UNIFIED_FEATURE_SETS:
                rows.extend(_run_single_tabular_ml_task(asset, feature_set, model_name))

    for model_name in SEQUENCE_ML_MODELS:
        debug_print(f'Running sequence model sequentially | model={model_name}')
        for asset in ASSETS:
            for feature_set in UNIFIED_FEATURE_SETS:
                rows.extend(_run_single_sequence_ml_task(asset, feature_set, model_name))

    for model_name in RULE_MODELS:
        debug_print(f'Running rule model sequentially | model={model_name}')
        for asset in ASSETS:
            for strategy_mode in UNIFIED_STRATEGY_MODES:
                for cost_label in COST_REGIMES.keys():
                    rows.extend(_run_single_rule_task(asset, model_name, strategy_mode, cost_label))

    unified_summary = pd.DataFrame(rows)
    unified_summary.to_csv(DATA_LOG_DIR / 'unified_experiment_summary.csv', index=False)
    print('Main experiment grid completed. Logs saved under data/trading_logs/.')
    display(unified_summary.head(20))
else:
    print('RUN_UNIFIED_GRID=False, skipping main experiment grid.')

E0000 00:00:1776882565.755404 5704139 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
E0000 00:00:1776882602.518263 5704139 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),

Main experiment grid completed. Logs saved under data/trading_logs/.


,Asset,Model,Model Family,Feature Set,Input Type,Feature Type,Strategy Mode,Strategy Label,Cost Regime,Long Threshold,...,Test Precision,Test Recall,Test Avg Return,Cumulative Return,Maximum Drawdown,Turnover,Number of Trades,Annualized Sharpe,Model Params,Log Path
0,BTC,Logistic Regression,ML,ohlc_raw,ohlc,raw,long_only,long,with_cost,0.59,...,0.611570,0.013165,0.000902,0.075107,-0.098622,172.0,172.0,0.615844,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
1,BTC,Logistic Regression,ML,ohlc_raw,ohlc,raw,long_only,long,no_cost,0.59,...,0.611570,0.013165,0.001151,0.141810,-0.090344,172.0,172.0,1.084217,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
2,BTC,Logistic Regression,ML,ohlc_raw,ohlc,raw,short_only,short,with_cost,NaN,...,0.500294,0.638972,-0.000092,-0.666037,-0.686616,2287.0,2287.0,-2.410181,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
3,BTC,Logistic Regression,ML,ohlc_raw,ohlc,raw,short_only,short,no_cost,NaN,...,0.500294,0.638972,-0.000033,-0.256217,-0.365644,2287.0,2287.0,-0.527697,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
4,BTC,Logistic Regression,ML,ohlc_raw,ohlc,raw,long_short,long_short,with_cost,0.55,...,0.285891,0.020548,0.000278,0.015911,-0.154200,456.0,456.0,0.159229,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
5,BTC,Logistic Regression,ML,ohlc_raw,ohlc,raw,long_short,long_short,no_cost,0.55,...,0.285891,0.020548,0.000474,0.191773,-0.127523,456.0,456.0,0.932774,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
6,BTC,Logistic Regression,ML,candle_raw,candle,raw,long_only,long,with_cost,0.52,...,0.536111,0.068671,0.000375,0.041210,-0.208967,1142.0,1142.0,0.257171,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
7,BTC,Logistic Regression,ML,candle_raw,candle,raw,long_only,long,no_cost,0.52,...,0.536111,0.068671,0.000652,0.552763,-0.122664,1142.0,1142.0,1.722623,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
8,BTC,Logistic Regression,ML,candle_raw,candle,raw,short_only,short,with_cost,NaN,...,0.521545,0.065829,0.000134,-0.061460,-0.137543,822.0,822.0,-0.335381,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...
9,BTC,Logistic Regression,ML,candle_raw,candle,raw,short_only,short,no_cost,NaN,...,0.521545,0.065829,0.000348,0.251388,-0.068247,822.0,822.0,1.472212,"{""model_dir"": ""/Users/wuxuande/PycharmProjects...",/Users/wuxuande/PycharmProjects/QF4211Project/...


## 8. Tables and Aggregated Comparisons
This section converts the full experiment grid into report-ready tables.

The reporting flow now separates configurations into two buckets:
- `feasible`: thresholds passed the validation guardrails and are used for the main analysis,
- `infeasible`: thresholds were still exported in the raw experiment output, but they are treated as diagnostics rather than headline results.

Main exports in this section therefore focus on feasible configurations, while separate diagnostics are written for infeasible rows.


In [ ]:
# -- Reporting helper definitions ----------------------
if 'unified_summary' not in globals() or unified_summary is None or len(unified_summary) == 0:
    p = DATA_LOG_DIR / 'unified_experiment_summary.csv'
    unified_summary = pd.read_csv(p) if p.exists() else pd.DataFrame()


def _sort_detail_table(df):
    return df.sort_values(['Test Precision', 'Validation Precision', 'Test Sharpe'], ascending=[False, False, False]).reset_index(drop=True)


def _paper_metric_summary(df, group_cols):
    metric_cols = [
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Test Sharpe', 'Test Precision', 'Test Recall', 'Test Avg Return',
    ]
    return (df.groupby(group_cols, as_index=False)[metric_cols]
              .mean()
              .sort_values(group_cols)
              .reset_index(drop=True))


def _normalize_threshold_feasible(values):
    def _coerce(value):
        if pd.isna(value):
            return False
        if isinstance(value, (bool, np.bool_)):
            return bool(value)
        text = str(value).strip().lower()
        if text in {'true', '1', 'yes', 'y'}:
            return True
        if text in {'false', '0', 'no', 'n', 'nan', 'none', ''}:
            return False
        return bool(value)

    return pd.Series(values).map(_coerce).astype(bool)


def _rank_configs(df, dedupe_cols):
    ranked = df.copy()
    for col in ['Validation Precision', 'Validation Recall', 'Validation Sharpe', 'Test Sharpe', 'Test Avg Return']:
        ranked[col] = pd.to_numeric(ranked[col], errors='coerce')
    ranked = ranked.sort_values(
        dedupe_cols + ['Validation Precision', 'Validation Recall', 'Validation Sharpe', 'Test Sharpe', 'Test Avg Return'],
        ascending=[True] * len(dedupe_cols) + [False, False, False, False, False],
        na_position='last',
    )
    return ranked.drop_duplicates(subset=dedupe_cols, keep='first').reset_index(drop=True)


def _select_best_feature_configs(df):
    return _rank_configs(df, ['Asset', 'Model', 'Feature Set', 'Strategy Mode', 'Cost Regime'])


def _select_best_model_configs(df):
    return _rank_configs(df, ['Asset', 'Model', 'Strategy Mode', 'Cost Regime'])


def _select_best_overall_configs(df):
    return _rank_configs(df, ['Asset', 'Strategy Mode', 'Cost Regime'])


def _iter_scopes(df):
    yield 'all_assets', df.copy()
    for asset in ASSETS:
        asset_df = df[df['Asset'] == asset].copy()
        if len(asset_df) > 0:
            yield asset.lower(), asset_df


def _strategy_table_tag(strategy_mode):
    return {
        'long_only': 'table8',
        'short_only': 'table9',
        'long_short': 'table_long_short',
    }[strategy_mode]


def _paper_detail_view(df, include_asset=False):
    cols = (['Asset'] if include_asset else []) + [
        'Model', 'Input Type', 'Feature Type', 'Strategy Label',
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Test Sharpe', 'Test Precision', 'Test Recall', 'Test Avg Return',
    ]
    out = df.reindex(columns=cols).rename(columns={'Strategy Label': 'Strategy'}).copy()
    sort_cols = (['Asset'] if include_asset else []) + ['Test Precision', 'Validation Precision', 'Test Sharpe']
    ascending = ([True] if include_asset else []) + [False, False, False]
    return out.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)


def _infeasible_overview(df):
    cols = [
        'Asset', 'Model', 'Model Family', 'Feature Set', 'Strategy Mode', 'Cost Regime',
        'Validation Trades', 'Validation Signal Ratio', 'Validation Precision', 'Validation Recall', 'Validation Sharpe',
        'Test Precision', 'Test Recall', 'Test Sharpe', 'Number of Trades', 'Log Path',
    ]
    return df.reindex(columns=cols).sort_values(['Asset', 'Model', 'Strategy Mode', 'Cost Regime', 'Feature Set']).reset_index(drop=True)

In [ ]:
# -- Table execution ------------------------------------
if len(unified_summary) == 0:
    print('No unified results available yet.')
    comparison = pd.DataFrame()
    comparison_feasible = pd.DataFrame()
    comparison_infeasible = pd.DataFrame()
    best_feature_configs = pd.DataFrame()
    best_model_configs = pd.DataFrame()
    best_overall_configs = pd.DataFrame()
else:
    comparison = unified_summary.copy()
    numeric_cols = [
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Validation Trades', 'Validation Signal Ratio', 'Test Sharpe', 'Test Precision',
        'Test Recall', 'Test Avg Return', 'Cumulative Return', 'Maximum Drawdown',
        'Turnover', 'Number of Trades', 'Annualized Sharpe',
    ]
    for col in numeric_cols:
        if col in comparison.columns:
            comparison[col] = pd.to_numeric(comparison[col], errors='coerce')

    if 'Model Family' not in comparison.columns:
        comparison['Model Family'] = np.where(comparison['Model'].isin(RULE_MODELS), 'Rule', 'ML')
    if 'Input Type' not in comparison.columns and 'Feature Set' in comparison.columns:
        comparison['Input Type'] = np.where(comparison['Feature Set'].astype(str).str.startswith('candle'), 'candle', 'ohlc')
    if 'Feature Type' not in comparison.columns and 'Feature Set' in comparison.columns:
        comparison['Feature Type'] = np.where(comparison['Feature Set'].astype(str).str.endswith('extended'), 'extended', 'raw')
    if 'Strategy Label' not in comparison.columns and 'Strategy Mode' in comparison.columns:
        comparison['Strategy Label'] = comparison['Strategy Mode'].map({
            'long_only': 'long',
            'short_only': 'short',
            'long_short': 'long_short',
        })
    if 'Threshold Feasible' not in comparison.columns:
        comparison['Threshold Feasible'] = False
    if 'Annualized Sharpe' not in comparison.columns:
        comparison['Annualized Sharpe'] = np.nan
    if 'Model Params' not in comparison.columns:
        comparison['Model Params'] = '{}'

    comparison['Threshold Feasible'] = _normalize_threshold_feasible(comparison['Threshold Feasible'])
    comparison['Feasibility Bucket'] = np.where(comparison['Threshold Feasible'], 'feasible', 'infeasible')
    comparison = comparison.sort_values(['Asset', 'Model Family', 'Model', 'Feature Set', 'Strategy Mode', 'Cost Regime']).reset_index(drop=True)
    comparison.to_csv(RESULT_DIR / 'summary_all_strategies_raw.csv', index=False)

    comparison_feasible = comparison[comparison['Threshold Feasible']].copy().reset_index(drop=True)
    comparison_infeasible = comparison[~comparison['Threshold Feasible']].copy().reset_index(drop=True)
    comparison_feasible.to_csv(RESULT_DIR / 'summary_all_strategies_feasible.csv', index=False)
    comparison_infeasible.to_csv(RESULT_DIR / 'summary_all_strategies_infeasible.csv', index=False)

    detail_cols = [
        'Asset', 'Model', 'Model Family', 'Input Type', 'Feature Type', 'Feature Set', 'Strategy Label',
        'Cost Regime', 'Threshold Feasible', 'Feasibility Bucket',
        'Validation Sharpe', 'Validation Precision', 'Validation Recall', 'Validation Avg Return',
        'Validation Trades', 'Validation Signal Ratio',
        'Test Sharpe', 'Test Precision', 'Test Recall', 'Test Avg Return', 'Cumulative Return',
        'Maximum Drawdown', 'Turnover', 'Number of Trades', 'Annualized Sharpe', 'Model Params',
    ]
    detail_all = _sort_detail_table(comparison.reindex(columns=detail_cols))
    detail_all.to_csv(RESULT_DIR / 'paper_tables' / 'table_detail_all_configs.csv', index=False)

    detail_feasible = _sort_detail_table(comparison_feasible.reindex(columns=detail_cols)) if len(comparison_feasible) > 0 else pd.DataFrame(columns=detail_cols)
    detail_feasible.to_csv(RESULT_DIR / 'paper_tables' / 'table_detail_feasible_configs.csv', index=False)

    detail_infeasible = _infeasible_overview(comparison_infeasible) if len(comparison_infeasible) > 0 else pd.DataFrame()
    detail_infeasible.to_csv(RESULT_DIR / 'paper_tables' / 'table_detail_infeasible_configs.csv', index=False)

    if len(comparison_infeasible) > 0:
        infeasible_counts = (comparison_infeasible.groupby(['Asset', 'Model', 'Strategy Mode', 'Cost Regime'], as_index=False)
                             .size()
                             .rename(columns={'size': 'Infeasible Config Count'})
                             .sort_values(['Asset', 'Model', 'Strategy Mode', 'Cost Regime']))
        infeasible_counts.to_csv(RESULT_DIR / 'paper_tables' / 'infeasible_config_counts.csv', index=False)
    else:
        infeasible_counts = pd.DataFrame()

    grouped_exports = {'table10_feasible': [], 'table11_feasible': [], 'table12_feasible': []}
    preview_tables = []

    for scope_name, scope_df in _iter_scopes(comparison_feasible):
        include_asset = scope_name == 'all_assets'
        for strategy_mode in UNIFIED_STRATEGY_MODES:
            for cost_regime in COST_REGIMES.keys():
                sub = scope_df[(scope_df['Strategy Mode'] == strategy_mode) & (scope_df['Cost Regime'] == cost_regime)].copy()
                if len(sub) == 0:
                    continue

                detail_view = _paper_detail_view(sub, include_asset=include_asset)
                detail_name = f'{_strategy_table_tag(strategy_mode)}_{scope_name}_{cost_regime}_feasible.csv'
                detail_view.to_csv(RESULT_DIR / 'paper_tables' / detail_name, index=False)

                ml_sub = sub[sub['Model Family'] == 'ML'].copy()
                if len(ml_sub) > 0:
                    table10 = _paper_metric_summary(ml_sub, ['Input Type'])
                    table10.to_csv(RESULT_DIR / 'paper_tables' / f'table10_representation_{scope_name}_{strategy_mode}_{cost_regime}_feasible.csv', index=False)
                    grouped_exports['table10_feasible'].append(table10.assign(Scope=scope_name, Strategy_Mode=strategy_mode, Cost_Regime=cost_regime))
                    if len(preview_tables) < 4:
                        preview_tables.append((f'Table 10 | {scope_name} | {strategy_mode} | {cost_regime} | feasible', table10))

                    table11 = _paper_metric_summary(ml_sub, ['Feature Type'])
                    table11.to_csv(RESULT_DIR / 'paper_tables' / f'table11_feature_type_{scope_name}_{strategy_mode}_{cost_regime}_feasible.csv', index=False)
                    grouped_exports['table11_feasible'].append(table11.assign(Scope=scope_name, Strategy_Mode=strategy_mode, Cost_Regime=cost_regime))
                    if len(preview_tables) < 6:
                        preview_tables.append((f'Table 11 | {scope_name} | {strategy_mode} | {cost_regime} | feasible', table11))

                    table12 = _paper_metric_summary(sub, ['Model'])
                    table12.to_csv(RESULT_DIR / 'paper_tables' / f'table12_model_average_{scope_name}_{strategy_mode}_{cost_regime}_feasible.csv', index=False)
                    grouped_exports['table12_feasible'].append(table12.assign(Scope=scope_name, Strategy_Mode=strategy_mode, Cost_Regime=cost_regime))

    for name, frames in grouped_exports.items():
        if len(frames) > 0:
            pd.concat(frames, ignore_index=True).to_csv(RESULT_DIR / 'paper_tables' / f'{name}_bundle.csv', index=False)

    cost_compare = comparison_feasible.pivot_table(
        index=['Asset', 'Model', 'Model Family', 'Feature Set', 'Input Type', 'Feature Type', 'Strategy Mode'],
        columns='Cost Regime',
        values=['Test Sharpe', 'Test Avg Return', 'Cumulative Return', 'Turnover', 'Number of Trades'],
    ) if len(comparison_feasible) > 0 else pd.DataFrame()
    if len(cost_compare) > 0:
        cost_compare.columns = [' '.join(col).strip() for col in cost_compare.columns.to_flat_index()]
        cost_compare = cost_compare.reset_index()
        if 'Test Sharpe no_cost' in cost_compare.columns and 'Test Sharpe with_cost' in cost_compare.columns:
            cost_compare['Sharpe Cost Drag'] = cost_compare['Test Sharpe no_cost'] - cost_compare['Test Sharpe with_cost']
        if 'Test Avg Return no_cost' in cost_compare.columns and 'Test Avg Return with_cost' in cost_compare.columns:
            cost_compare['Avg Return Cost Drag'] = cost_compare['Test Avg Return no_cost'] - cost_compare['Test Avg Return with_cost']
        if 'Cumulative Return no_cost' in cost_compare.columns and 'Cumulative Return with_cost' in cost_compare.columns:
            cost_compare['Cumulative Return Cost Drag'] = cost_compare['Cumulative Return no_cost'] - cost_compare['Cumulative Return with_cost']
    cost_compare.to_csv(RESULT_DIR / 'paper_tables' / 'cost_comparison_by_configuration_feasible.csv', index=False)

    best_feature_configs = _select_best_feature_configs(comparison_feasible) if len(comparison_feasible) > 0 else pd.DataFrame()
    best_model_configs = _select_best_model_configs(comparison_feasible) if len(comparison_feasible) > 0 else pd.DataFrame()
    best_overall_configs = _select_best_overall_configs(comparison_feasible) if len(comparison_feasible) > 0 else pd.DataFrame()
    invalid_model_configs = _select_best_model_configs(comparison_infeasible) if len(comparison_infeasible) > 0 else pd.DataFrame()

    best_feature_configs.to_csv(RESULT_DIR / 'paper_tables' / 'best_feature_configs_feasible.csv', index=False)
    best_model_configs.to_csv(RESULT_DIR / 'paper_tables' / 'best_model_configs_feasible.csv', index=False)
    best_overall_configs.to_csv(RESULT_DIR / 'paper_tables' / 'best_overall_configs_feasible.csv', index=False)
    invalid_model_configs.to_csv(RESULT_DIR / 'paper_tables' / 'best_model_configs_infeasible.csv', index=False)

    print('===== FEASIBLE PAPER-STYLE TABLES SAVED =====')
    display(detail_feasible.head(20))
    if len(detail_infeasible) > 0:
        print('===== INFEASIBLE CONFIGURATION DIAGNOSTICS =====')
        display(detail_infeasible.head(20))
    print('===== SAMPLE GROUPED TABLE OUTPUTS =====')
    for title, tbl in preview_tables:
        print(title)
        display(tbl)

===== FEASIBLE PAPER-STYLE TABLES SAVED =====


,Asset,Model,Model Family,Input Type,Feature Type,Feature Set,Strategy Label,Cost Regime,Threshold Feasible,Feasibility Bucket,...,Test Sharpe,Test Precision,Test Recall,Test Avg Return,Cumulative Return,Maximum Drawdown,Turnover,Number of Trades,Annualized Sharpe,Model Params
0,BTC,Logistic Regression,ML,ohlc,extended,ohlc_extended,long,no_cost,True,feasible,...,0.260394,0.700000,0.007472,0.002032,0.127337,-0.033085,92.0,92.0,1.689482,"{""model_dir"": ""/Users/wuxuande/PycharmProjects..."
1,BTC,Logistic Regression,ML,ohlc,extended,ohlc_extended,long,with_cost,True,feasible,...,0.223585,0.700000,0.007472,0.001764,0.091643,-0.036137,92.0,92.0,1.251324,"{""model_dir"": ""/Users/wuxuande/PycharmProjects..."
2,ETH,Logistic Regression,ML,ohlc,raw,ohlc_raw,long_short,no_cost,True,feasible,...,0.007837,0.699018,0.129930,0.000053,0.111850,-0.252582,1832.0,1832.0,0.468427,"{""model_dir"": ""/Users/wuxuande/PycharmProjects..."
3,ETH,Logistic Regression,ML,ohlc,raw,ohlc_raw,long_short,with_cost,True,feasible,...,-0.013531,0.699018,0.129930,-0.000066,-0.414481,-0.514931,1832.0,1832.0,-1.616238,"{""model_dir"": ""/Users/wuxuande/PycharmProjects..."
4,ETH,GRU,ML,ohlc,extended,ohlc_extended,long_short,no_cost,True,feasible,...,0.012247,0.686186,0.018440,0.000162,0.040329,-0.233564,464.0,464.0,0.279586,"{""lookback_hours"": 24, ""model_dir"": ""/Users/wu..."
5,ETH,GRU,ML,ohlc,extended,ohlc_extended,long_short,with_cost,True,feasible,...,-0.012387,0.686186,0.018440,-0.000075,-0.115688,-0.284944,464.0,464.0,-0.536642,"{""lookback_hours"": 24, ""model_dir"": ""/Users/wu..."
6,ETH,Logistic Regression,ML,candle,extended,candle_extended,long_short,no_cost,True,feasible,...,0.043044,0.672659,0.015191,0.000472,0.128474,-0.121359,522.0,521.0,0.695693,"{""model_dir"": ""/Users/wuxuande/PycharmProjects..."
7,ETH,Logistic Regression,ML,candle,extended,candle_extended,long_short,with_cost,True,feasible,...,0.009881,0.672659,0.015191,0.000155,-0.060016,-0.174010,522.0,521.0,-0.235133,"{""model_dir"": ""/Users/wuxuande/PycharmProjects..."
8,BTC,GRU,ML,ohlc,extended,ohlc_extended,long,no_cost,True,feasible,...,0.244025,0.653061,0.005704,0.001433,0.071748,-0.032590,84.0,84.0,1.469876,"{""lookback_hours"": 24, ""model_dir"": ""/Users/wu..."
9,BTC,GRU,ML,ohlc,extended,ohlc_extended,long,with_cost,True,feasible,...,0.190682,0.653061,0.005704,0.001133,0.040712,-0.034585,84.0,84.0,0.861124,"{""lookback_hours"": 24, ""model_dir"": ""/Users/wu..."


===== SAMPLE GROUPED TABLE OUTPUTS =====
Table 10 | all_assets | long_only | with_cost | feasible


,Input Type,Validation Sharpe,Validation Precision,Validation Recall,Validation Avg Return,Test Sharpe,Test Precision,Test Recall,Test Avg Return
0,candle,0.022155,0.568749,0.104255,0.000085,0.054389,0.548725,0.086410,0.000612
1,ohlc,0.005242,0.564779,0.115164,0.000081,0.045366,0.553358,0.117458,0.000507


Table 11 | all_assets | long_only | with_cost | feasible


...

,Feature Type,Validation Sharpe,Validation Precision,Validation Recall,Validation Avg Return,Test Sharpe,Test Precision,Test Recall,Test Avg Return
0,extended,0.054575,0.606635,0.058284,0.000429,0.119895,0.582835,0.061023,0.001213
1,raw,0.031659,0.526893,0.161136,0.000209,0.041041,0.519248,0.142845,0.000379


Table 11 | all_assets | short_only | with_cost | feasible


,Feature Type,Validation Sharpe,Validation Precision,Validation Recall,Validation Avg Return,Test Sharpe,Test Precision,Test Recall,Test Avg Return
0,extended,-0.026884,0.580801,0.061368,-0.000213,-0.023805,0.545011,0.139516,-0.000231
1,raw,-0.032713,0.515126,0.137697,-0.000366,-0.020572,0.495674,0.324543,-0.000287


Table 11 | all_assets | short_only | no_cost | feasible


,Feature Type,Validation Sharpe,Validation Precision,Validation Recall,Validation Avg Return,Test Sharpe,Test Precision,Test Recall,Test Avg Return
0,extended,0.004552,0.580801,0.061368,0.000054,0.004816,0.545011,0.139516,0.000003
1,raw,-0.006180,0.515126,0.137697,-0.000151,-0.003614,0.495674,0.324543,-0.000130


## 9. Graphs and Statistical Tests
This section separates visualization from inference.

Main graphs and headline statistical tests are run on `feasible` configurations only. Infeasible configurations remain available in the exported diagnostics, but they are excluded from the main charts and hypothesis tests so they do not dilute the interpretation.


In [ ]:
# -- Plotting and stat-test helpers ---------------------
if 'comparison' not in globals() or comparison is None or len(comparison) == 0:
    p = DATA_LOG_DIR / 'unified_experiment_summary.csv'
    comparison = pd.read_csv(p) if p.exists() else pd.DataFrame()
if 'comparison_feasible' not in globals() or comparison_feasible is None:
    comparison_feasible = comparison[comparison['Threshold Feasible']].copy() if len(comparison) > 0 and 'Threshold Feasible' in comparison.columns else pd.DataFrame()
if 'comparison_infeasible' not in globals() or comparison_infeasible is None:
    comparison_infeasible = comparison[~comparison['Threshold Feasible']].copy() if len(comparison) > 0 and 'Threshold Feasible' in comparison.columns else pd.DataFrame()
if 'best_feature_configs' not in globals() or best_feature_configs is None or len(best_feature_configs) == 0:
    best_feature_configs = _select_best_feature_configs(comparison_feasible) if len(comparison_feasible) > 0 else pd.DataFrame()
if 'best_model_configs' not in globals() or best_model_configs is None or len(best_model_configs) == 0:
    best_model_configs = _select_best_model_configs(comparison_feasible) if len(comparison_feasible) > 0 else pd.DataFrame()
if 'best_overall_configs' not in globals() or best_overall_configs is None or len(best_overall_configs) == 0:
    best_overall_configs = _select_best_overall_configs(comparison_feasible) if len(comparison_feasible) > 0 else pd.DataFrame()


def _load_log_returns(log_path):
    lp = Path(log_path)
    if not lp.exists():
        return None
    df = pd.read_csv(lp)
    if 'datetime' not in df.columns or 'net_return' not in df.columns:
        return None
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True, errors='coerce')
    out = df[['datetime', 'net_return']].dropna(subset=['datetime', 'net_return']).copy()
    return out.sort_values('datetime').drop_duplicates('datetime').reset_index(drop=True)


def _group_mean_returns(df):
    merged = None
    series_count = 0
    for _, row in df.iterrows():
        s = _load_log_returns(row['Log Path'])
        if s is None or len(s) == 0:
            continue
        work = s.rename(columns={'net_return': f'net_return_{series_count}'})
        merged = work if merged is None else merged.merge(work, on='datetime', how='inner')
        series_count += 1
    if merged is None or len(merged) == 0:
        return pd.DataFrame(columns=['datetime', 'net_return'])
    value_cols = [c for c in merged.columns if c != 'datetime']
    return merged[['datetime']].assign(net_return=merged[value_cols].mean(axis=1))


def _scope_subset(df, scope_name):
    if scope_name == 'all_assets':
        return df.copy()
    return df[df['Asset'] == scope_name.upper()].copy()


def _group_bootstrap_task(source_df, scope_name, strategy_mode, cost_regime, test_type, population_label):
    scope_df = _scope_subset(source_df, scope_name)
    ml_sub = scope_df[
        (scope_df['Model Family'] == 'ML') &
        (scope_df['Strategy Mode'] == strategy_mode) &
        (scope_df['Cost Regime'] == cost_regime)
    ].copy()
    if len(ml_sub) == 0:
        return None

    if test_type == 'candle_vs_ohlc':
        left = _group_mean_returns(ml_sub[ml_sub['Input Type'] == 'candle'])
        right = _group_mean_returns(ml_sub[ml_sub['Input Type'] == 'ohlc'])
    elif test_type == 'extended_vs_raw':
        left = _group_mean_returns(ml_sub[ml_sub['Feature Type'] == 'extended'])
        right = _group_mean_returns(ml_sub[ml_sub['Feature Type'] == 'raw'])
    elif test_type == 'gru_vs_other_models':
        left = _group_mean_returns(ml_sub[ml_sub['Model'] == 'GRU'])
        right = _group_mean_returns(ml_sub[ml_sub['Model'] != 'GRU'])
    else:
        raise ValueError(f'Unsupported grouped bootstrap test: {test_type}')

    if len(left) == 0 or len(right) == 0:
        return None

    aligned = left.merge(right, on='datetime', how='inner', suffixes=('_left', '_right'))
    if len(aligned) == 0:
        return None

    res = block_bootstrap_mean_diff(aligned['net_return_left'], aligned['net_return_right'])
    return {
        'Population': population_label,
        'Scope': scope_name,
        'Test Type': test_type,
        'Strategy Mode': strategy_mode,
        'Cost Regime': cost_regime,
        'Mean Net Return Diff': res['mean_diff'],
        'Bootstrap p-value': res['p_value'],
        'Block Length': res['block_length'],
        'Bootstrap Iterations': res['n_boot'],
    }


def _usefulness_task(row, population_label):
    strat_ret = _load_log_returns(row['Log Path'])
    if strat_ret is None or len(strat_ret) == 0:
        return None
    zero_test = block_bootstrap_mean_vs_zero(strat_ret['net_return'])
    bh_bt = backtest_buy_and_hold(test_data[row['Asset']], transaction_cost=COST_REGIMES[row['Cost Regime']])
    bh_ret = bh_bt[['datetime', 'net_return']].dropna(subset=['datetime', 'net_return']).copy()
    bh_ret['datetime'] = pd.to_datetime(bh_ret['datetime'], utc=True, errors='coerce')
    bh_ret = bh_ret.sort_values('datetime').drop_duplicates('datetime').reset_index(drop=True)
    aligned = strat_ret.merge(bh_ret, on='datetime', how='inner', suffixes=('_strategy', '_buyhold'))
    if len(aligned) == 0:
        return None
    diff_test = block_bootstrap_mean_diff(aligned['net_return_strategy'], aligned['net_return_buyhold'], alternative='greater')
    return {
        'Population': population_label,
        'Asset': row['Asset'],
        'Model': row['Model'],
        'Feature Set': row['Feature Set'],
        'Strategy Mode': row['Strategy Mode'],
        'Cost Regime': row['Cost Regime'],
        'Mean Net Return': zero_test['mean'],
        'p-value vs 0': zero_test['p_value'],
        'Mean Excess Return vs BuyHold': diff_test['mean_diff'],
        'p-value vs BuyHold': diff_test['p_value'],
        'Block Length': zero_test['block_length'],
        'Bootstrap Iterations': zero_test['n_boot'],
    }


def _graph_output_dir(asset, cost_regime, strategy_mode, population_label='feasible'):
    out_dir = RESULT_DIR / 'graph' / population_label / asset / cost_regime / strategy_mode
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def _plot_asset_equity_curves(asset, strategy_mode, cost_regime, configs, population_label='feasible'):
    sub = configs[
        (configs['Asset'] == asset) &
        (configs['Strategy Mode'] == strategy_mode) &
        (configs['Cost Regime'] == cost_regime)
    ].copy()
    if len(sub) == 0:
        return None

    sub = sub.sort_values(['Model Family', 'Model', 'Validation Precision'], ascending=[True, True, False])
    fig, ax = plt.subplots(figsize=(12, 7))
    plotted = 0
    for _, row in sub.iterrows():
        lp = Path(row['Log Path'])
        if not lp.exists():
            continue
        bt = pd.read_csv(lp)
        if 'net_equity_curve' not in bt.columns:
            continue
        label = f"{row['Model']} [{row['Feature Set']}]"
        ax.plot(bt['net_equity_curve'].values, linewidth=1.3, label=label)
        plotted += 1

    bh_bt = backtest_buy_and_hold(test_data[asset], transaction_cost=COST_REGIMES[cost_regime])
    ax.plot(bh_bt['net_equity_curve'].values, color='black', linestyle='--', linewidth=1.6, label='Buy&Hold')
    ax.set_title(f"{asset} | {strategy_mode} | {cost_regime} | best feature per model | {population_label}")
    ax.set_ylabel('Equity (start=1)')
    ax.set_xlabel('Hour')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.25)
    plt.tight_layout()

    out_dir = _graph_output_dir(asset, cost_regime, strategy_mode, population_label=population_label)
    out_path = out_dir / f'equity_curves_best_feature_per_model_{population_label}.png'
    plt.savefig(out_path, dpi=140)
    plt.close(fig)
    return out_path if plotted > 0 else None

In [ ]:
# -- Plotting and stat-test execution -------------------
if len(best_feature_configs) == 0:
    print('No feasible configurations available for the main graphs and statistical tests.')
else:
    saved_graphs = []
    for asset in ASSETS:
        for cost_regime in COST_REGIMES.keys():
            for strategy_mode in UNIFIED_STRATEGY_MODES:
                out_path = _plot_asset_equity_curves(asset, strategy_mode, cost_regime, best_model_configs, population_label='feasible')
                if out_path is not None:
                    saved_graphs.append(str(out_path))

    scope_names = ['all_assets'] + [asset.lower() for asset in ASSETS]
    grouped_tasks = [
        (scope_name, strategy_mode, cost_regime, test_type)
        for scope_name in scope_names
        for strategy_mode in UNIFIED_STRATEGY_MODES
        for cost_regime in COST_REGIMES.keys()
        for test_type in ['candle_vs_ohlc', 'extended_vs_raw', 'gru_vs_other_models']
    ]
    bootstrap_rows = []
    for scope_name, strategy_mode, cost_regime, test_type in grouped_tasks:
        row = _group_bootstrap_task(comparison_feasible, scope_name, strategy_mode, cost_regime, test_type, population_label='feasible')
        if row is not None:
            bootstrap_rows.append(row)

    usefulness_rows = []
    for row in best_overall_configs.to_dict('records'):
        result = _usefulness_task(row, population_label='feasible')
        if result is not None:
            usefulness_rows.append(result)

    bootstrap_df = pd.DataFrame(bootstrap_rows)
    usefulness_df = pd.DataFrame(usefulness_rows)
    bootstrap_df.to_csv(RESULT_DIR / 'stat_tests' / 'paper_group_bootstrap_tests_feasible.csv', index=False)
    usefulness_df.to_csv(RESULT_DIR / 'stat_tests' / 'usefulness_tests_best_overall_feasible.csv', index=False)

    invalid_stats_manifest = comparison_infeasible.reindex(columns=[
        'Asset', 'Model', 'Model Family', 'Feature Set', 'Strategy Mode', 'Cost Regime',
        'Validation Trades', 'Validation Signal Ratio', 'Validation Precision', 'Validation Recall', 'Validation Sharpe',
        'Log Path',
    ]).copy() if len(comparison_infeasible) > 0 else pd.DataFrame()
    invalid_stats_manifest.to_csv(RESULT_DIR / 'stat_tests' / 'infeasible_configurations_excluded_from_main_stats.csv', index=False)

    print(f'Saved {len(saved_graphs)} feasible equity-curve graphs under result/graph/feasible/.')
    print('Saved feasible grouped bootstrap tests and usefulness tests under result/stat_tests/.')
    if len(invalid_stats_manifest) > 0:
        print(f'Excluded {len(invalid_stats_manifest)} infeasible configurations from the main graphs and tests; diagnostics were saved separately.')
    display(pd.DataFrame({'Graph Path': saved_graphs}).head(24))
    display(bootstrap_df.head(20))
    display(usefulness_df.head(30))

Saved 24 feasible equity-curve graphs under result/graph/feasible/.
Saved feasible grouped bootstrap tests and usefulness tests under result/stat_tests/.


,Graph Path
0,/Users/wuxuande/PycharmProjects/QF4211Project/...
1,/Users/wuxuande/PycharmProjects/QF4211Project/...
2,/Users/wuxuande/PycharmProjects/QF4211Project/...
3,/Users/wuxuande/PycharmProjects/QF4211Project/...
4,/Users/wuxuande/PycharmProjects/QF4211Project/...
5,/Users/wuxuande/PycharmProjects/QF4211Project/...
6,/Users/wuxuande/PycharmProjects/QF4211Project/...
7,/Users/wuxuande/PycharmProjects/QF4211Project/...
8,/Users/wuxuande/PycharmProjects/QF4211Project/...
9,/Users/wuxuande/PycharmProjects/QF4211Project/...


,Population,Scope,Test Type,Strategy Mode,Cost Regime,Mean Net Return Diff,Bootstrap p-value,Block Length,Bootstrap Iterations
0,feasible,all_assets,candle_vs_ohlc,long_only,with_cost,1.539627e-06,0.8340,21,10000
1,feasible,all_assets,extended_vs_raw,long_only,with_cost,2.924704e-05,0.0055,21,10000
2,feasible,all_assets,gru_vs_other_models,long_only,with_cost,2.674782e-05,0.0005,21,10000
3,feasible,all_assets,candle_vs_ohlc,long_only,no_cost,3.905960e-07,0.9566,21,10000
4,feasible,all_assets,extended_vs_raw,long_only,no_cost,7.308078e-06,0.4928,21,10000
5,feasible,all_assets,gru_vs_other_models,long_only,no_cost,5.774378e-06,0.4247,21,10000
6,feasible,all_assets,candle_vs_ohlc,short_only,with_cost,-2.607425e-05,0.0004,21,10000
7,feasible,all_assets,extended_vs_raw,short_only,with_cost,4.689444e-05,0.0025,21,10000
8,feasible,all_assets,gru_vs_other_models,short_only,with_cost,7.760338e-06,0.6096,21,10000
9,feasible,all_assets,candle_vs_ohlc,short_only,no_cost,-1.365306e-05,0.0638,21,10000


,Population,Asset,Model,Feature Set,Strategy Mode,Cost Regime,Mean Net Return,p-value vs 0,Mean Excess Return vs BuyHold,p-value vs BuyHold,Block Length,Bootstrap Iterations
0,feasible,BTC,LightGBM,candle_extended,long_only,no_cost,0.000022,0.0023,-0.000101,0.9768,22,10000
1,feasible,BTC,LightGBM,candle_extended,long_only,with_cost,0.000015,0.0255,-0.000108,0.9848,22,10000
2,feasible,BTC,Logistic Regression,candle_extended,long_short,no_cost,0.000034,0.0271,-0.000089,0.9642,22,10000
3,feasible,BTC,Logistic Regression,candle_extended,long_short,with_cost,0.000015,0.1749,-0.000107,0.9862,22,10000
4,feasible,BTC,Logistic Regression,candle_extended,short_only,no_cost,0.000012,0.1068,-0.000110,0.9805,22,10000
5,feasible,BTC,Logistic Regression,candle_extended,short_only,with_cost,-0.000008,0.7739,-0.000130,0.9919,22,10000
6,feasible,ETH,LightGBM,ohlc_extended,long_only,no_cost,0.000016,0.0943,-0.000066,0.8636,22,10000
7,feasible,ETH,LightGBM,ohlc_extended,long_only,with_cost,0.000008,0.2765,-0.000074,0.8912,22,10000
8,feasible,ETH,LightGBM,ohlc_extended,long_short,no_cost,0.000024,0.0746,-0.000058,0.8356,22,10000
9,feasible,ETH,LightGBM,ohlc_extended,long_short,with_cost,0.000006,0.3629,-0.000076,0.8957,22,10000


## 10. Supplementary Cost and Feasibility Diagnostics
The final section gathers supporting material that is useful for the report but should not obscure the main interpretation:
- grouped table bundles derived from feasible configurations,
- cost-drag summaries by asset, model, and strategy mode,
- best-configuration overview tables for the feasible headline analysis,
- explicit diagnostics for infeasible configurations kept in the raw experiment output.


In [ ]:
if 'comparison' not in globals() or comparison is None or len(comparison) == 0:
    p = DATA_LOG_DIR / 'unified_experiment_summary.csv'
    comparison = pd.read_csv(p) if p.exists() else pd.DataFrame()
if 'comparison_feasible' not in globals() or comparison_feasible is None:
    comparison_feasible = comparison[comparison['Threshold Feasible']].copy() if len(comparison) > 0 and 'Threshold Feasible' in comparison.columns else pd.DataFrame()
if 'comparison_infeasible' not in globals() or comparison_infeasible is None:
    comparison_infeasible = comparison[~comparison['Threshold Feasible']].copy() if len(comparison) > 0 and 'Threshold Feasible' in comparison.columns else pd.DataFrame()
if 'best_overall_configs' not in globals() or best_overall_configs is None or len(best_overall_configs) == 0:
    best_overall_configs = _select_best_overall_configs(comparison_feasible) if len(comparison_feasible) > 0 else pd.DataFrame()

if len(comparison_feasible) == 0:
    print('No feasible unified summary available for supplementary reporting outputs.')
    supplementary_tables = pd.DataFrame()
else:
    cost_drag = pd.read_csv(RESULT_DIR / 'paper_tables' / 'cost_comparison_by_configuration_feasible.csv') if (RESULT_DIR / 'paper_tables' / 'cost_comparison_by_configuration_feasible.csv').exists() else pd.DataFrame()
    if len(cost_drag) > 0:
        cost_drag_summary = (cost_drag.groupby(['Asset', 'Model', 'Strategy Mode'], as_index=False)
                             [['Sharpe Cost Drag', 'Avg Return Cost Drag', 'Cumulative Return Cost Drag']]
                             .mean()
                             .sort_values(['Asset', 'Strategy Mode', 'Sharpe Cost Drag'], ascending=[True, True, False]))
        cost_drag_summary.to_csv(RESULT_DIR / 'paper_tables' / 'cost_drag_summary_by_asset_model_strategy_feasible.csv', index=False)
        print('Average feasible cost drag by asset, model, and strategy mode:')
        display(cost_drag_summary.head(20))
    else:
        cost_drag_summary = pd.DataFrame()

    bundle_frames = []
    for bundle_name in ['table10_feasible_bundle.csv', 'table11_feasible_bundle.csv', 'table12_feasible_bundle.csv']:
        bundle_path = RESULT_DIR / 'paper_tables' / bundle_name
        if bundle_path.exists():
            bundle_df = pd.read_csv(bundle_path)
            bundle_df.insert(0, 'Bundle', bundle_name.replace('_bundle.csv', ''))
            bundle_frames.append(bundle_df)

    supplementary_tables = pd.concat(bundle_frames, ignore_index=True) if len(bundle_frames) > 0 else pd.DataFrame()
    if len(supplementary_tables) > 0:
        supplementary_tables.to_csv(RESULT_DIR / 'paper_tables' / 'grouped_table_overview_feasible.csv', index=False)
        print('Grouped feasible paper-style table overview saved under result/paper_tables/.')
        display(supplementary_tables.head(20))

    if len(best_overall_configs) > 0:
        best_cols = [
            'Asset', 'Model', 'Model Family', 'Feature Set', 'Strategy Mode', 'Cost Regime',
            'Validation Precision', 'Validation Recall', 'Validation Sharpe', 'Test Sharpe',
            'Test Precision', 'Test Recall', 'Test Avg Return', 'Threshold Feasible',
        ]
        best_config_overview = best_overall_configs.reindex(columns=best_cols).sort_values(
            ['Asset', 'Strategy Mode', 'Cost Regime']
        ).reset_index(drop=True)
        best_config_overview.to_csv(RESULT_DIR / 'paper_tables' / 'best_config_overview_feasible.csv', index=False)
        print('Representative feasible best-configuration overview saved under result/paper_tables/.')
        display(best_config_overview.head(20))

    infeasible_overview = _infeasible_overview(comparison_infeasible) if len(comparison_infeasible) > 0 else pd.DataFrame()
    infeasible_overview.to_csv(RESULT_DIR / 'paper_tables' / 'infeasible_config_overview.csv', index=False)
    if len(infeasible_overview) > 0:
        print('Infeasible configuration overview saved under result/paper_tables/.')
        display(infeasible_overview.head(20))

Average feasible cost drag by asset, model, and strategy mode:


,Asset,Model,Strategy Mode,Sharpe Cost Drag,Avg Return Cost Drag,Cumulative Return Cost Drag
15,BTC,XGBoost,long_only,0.052279,0.000242,0.869570
3,BTC,LightGBM,long_only,0.050632,0.000271,0.308616
6,BTC,Logistic Regression,long_only,0.036924,0.000274,0.200675
0,BTC,GRU,long_only,0.033503,0.000258,0.114038
9,BTC,RSI,long_only,0.026552,0.000077,1.106401
12,BTC,SMA,long_only,0.003571,0.000026,0.006375
4,BTC,LightGBM,long_short,0.054197,0.000282,0.457916
7,BTC,Logistic Regression,long_short,0.040739,0.000241,0.479816
16,BTC,XGBoost,long_short,0.038972,0.000271,1.140001
1,BTC,GRU,long_short,0.023982,0.000157,0.354721


Grouped feasible paper-style table overview saved under result/paper_tables/.


,Bundle,Input Type,Validation Sharpe,Validation Precision,Validation Recall,Validation Avg Return,Test Sharpe,Test Precision,Test Recall,Test Avg Return,Scope,Strategy_Mode,Cost_Regime,Feature Type,Model
0,table10_feasible,candle,0.022155,0.568749,0.104255,0.000085,0.054389,0.548725,0.086410,0.000612,all_assets,long_only,with_cost,NaN,NaN
1,table10_feasible,ohlc,0.005242,0.564779,0.115164,0.000081,0.045366,0.553358,0.117458,0.000507,all_assets,long_only,with_cost,NaN,NaN
2,table10_feasible,candle,0.052789,0.568749,0.104255,0.000328,0.087270,0.548725,0.086410,0.000865,all_assets,long_only,no_cost,NaN,NaN
3,table10_feasible,ohlc,0.033445,0.564779,0.115164,0.000310,0.073666,0.553358,0.117458,0.000727,all_assets,long_only,no_cost,NaN,NaN
4,table10_feasible,candle,-0.018843,0.548998,0.118870,-0.000139,-0.026286,0.516978,0.253677,-0.000330,all_assets,short_only,with_cost,NaN,NaN
5,table10_feasible,ohlc,-0.040754,0.546928,0.080195,-0.000440,-0.018090,0.523707,0.210382,-0.000188,all_assets,short_only,with_cost,NaN,NaN
6,table10_feasible,candle,0.008470,0.548998,0.118870,0.000098,-0.003657,0.516978,0.253677,-0.000133,all_assets,short_only,no_cost,NaN,NaN
7,table10_feasible,ohlc,-0.010098,0.546928,0.080195,-0.000195,0.004859,0.523707,0.210382,0.000006,all_assets,short_only,no_cost,NaN,NaN
8,table10_feasible,candle,0.006608,0.664200,0.108417,0.000043,0.021458,0.524879,0.125570,0.000188,all_assets,long_short,with_cost,NaN,NaN
9,table10_feasible,ohlc,0.003468,0.640365,0.084538,0.000096,0.020067,0.545157,0.110339,0.000198,all_assets,long_short,with_cost,NaN,NaN


Representative feasible best-configuration overview saved under result/paper_tables/.


,Asset,Model,Model Family,Feature Set,Strategy Mode,Cost Regime,Validation Precision,Validation Recall,Validation Sharpe,Test Sharpe,Test Precision,Test Recall,Test Avg Return,Threshold Feasible
0,BTC,LightGBM,ML,candle_extended,long_only,no_cost,0.717949,0.010083,0.324554,0.260704,0.651852,0.015656,0.001787,True
1,BTC,LightGBM,ML,candle_extended,long_only,with_cost,0.717949,0.010083,0.241485,0.210831,0.651852,0.015656,0.001481,True
2,BTC,Logistic Regression,ML,candle_extended,long_short,no_cost,0.839583,0.014952,0.134241,0.136340,0.522545,0.018872,0.001077,True
3,BTC,Logistic Regression,ML,candle_extended,long_short,with_cost,0.839583,0.014952,0.080379,0.091630,0.522545,0.018872,0.000777,True
4,BTC,Logistic Regression,ML,candle_extended,short_only,no_cost,0.711864,0.015573,0.151581,0.058876,0.583333,0.043323,0.000326,True
5,BTC,Logistic Regression,ML,candle_extended,short_only,with_cost,0.711864,0.015573,0.116141,0.006099,0.583333,0.043323,0.000050,True
6,ETH,LightGBM,ML,ohlc_extended,long_only,no_cost,0.666667,0.015365,0.128050,0.089787,0.614379,0.016861,0.001142,True
7,ETH,LightGBM,ML,ohlc_extended,long_only,with_cost,0.666667,0.015365,0.073268,0.062974,0.614379,0.016861,0.000842,True
8,ETH,LightGBM,ML,ohlc_extended,long_short,no_cost,0.803636,0.015821,0.126009,0.075229,0.632055,0.018261,0.000751,True
9,ETH,LightGBM,ML,ohlc_extended,long_short,with_cost,0.803636,0.015821,0.067924,0.042038,0.632055,0.018261,0.000464,True


## 11. Regime Comparison Exports
This final section compares the main `2021-2025` run with the sandbox `2020-2024` rerun. 
It exports Appendix-C-style tables and comparison figures into `result/regime_comparison/` inside the sandbox only.


In [ ]:
# -- Regime comparison exports -------------------------
SANDBOX_ROOT = Path.cwd().resolve()
MAIN_PROJECT_ROOT = SANDBOX_ROOT.parent
MAIN_RESULT_DIR = MAIN_PROJECT_ROOT / 'result'
REGIME_COMPARE_DIR = RESULT_DIR / 'regime_comparison'
REGIME_COMPARE_DIR.mkdir(parents=True, exist_ok=True)


def _read_csv_or_empty(path):
    path = Path(path)
    if not path.exists():
        print(f'Missing input for regime comparison: {path}')
        return pd.DataFrame()
    return pd.read_csv(path)


def _with_window(df, window_label):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    out['Window'] = window_label
    return out


def _save_df(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def _clean_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.25)


def _plot_grouped_bars(df, *, x_col, value_col, hue_col, title, ylabel, output_path, hue_order=None, x_order=None):
    if len(df) == 0:
        print(f'Skipping empty plot: {output_path.name}')
        return None

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    hue_order = hue_order or list(dict.fromkeys(df[hue_col].tolist()))
    x_order = x_order or list(dict.fromkeys(df[x_col].tolist()))
    pivot = df.pivot(index=x_col, columns=hue_col, values=value_col).reindex(index=x_order, columns=hue_order)

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(pivot.index))
    width = 0.8 / max(len(pivot.columns), 1)

    for idx, col in enumerate(pivot.columns):
        ax.bar(x + (idx - (len(pivot.columns) - 1) / 2) * width, pivot[col].values, width=width, label=col)

    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    _clean_axes(ax)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return output_path


window_specs = [
    ('2021-2025', MAIN_RESULT_DIR),
    ('2020-2024', RESULT_DIR),
]

feasible_frames = []
buy_hold_frames = []
group_test_frames = []
usefulness_frames = []
best_overall_frames = []

for window_label, result_root in window_specs:
    feasible_frames.append(_with_window(_read_csv_or_empty(result_root / 'summary_all_strategies_feasible.csv'), window_label))
    buy_hold_frames.append(_with_window(_read_csv_or_empty(result_root / 'buy_and_hold' / 'summary_all_assets.csv'), window_label))
    group_test_frames.append(_with_window(_read_csv_or_empty(result_root / 'stat_tests' / 'paper_group_bootstrap_tests_feasible.csv'), window_label))
    usefulness_frames.append(_with_window(_read_csv_or_empty(result_root / 'stat_tests' / 'usefulness_tests_best_overall_feasible.csv'), window_label))
    best_overall_frames.append(_with_window(_read_csv_or_empty(result_root / 'paper_tables' / 'best_overall_configs_feasible.csv'), window_label))

feasible_both = pd.concat([df for df in feasible_frames if len(df) > 0], ignore_index=True) if any(len(df) > 0 for df in feasible_frames) else pd.DataFrame()
buy_hold_both = pd.concat([df for df in buy_hold_frames if len(df) > 0], ignore_index=True) if any(len(df) > 0 for df in buy_hold_frames) else pd.DataFrame()
group_tests_both = pd.concat([df for df in group_test_frames if len(df) > 0], ignore_index=True) if any(len(df) > 0 for df in group_test_frames) else pd.DataFrame()
usefulness_both = pd.concat([df for df in usefulness_frames if len(df) > 0], ignore_index=True) if any(len(df) > 0 for df in usefulness_frames) else pd.DataFrame()
best_overall_both = pd.concat([df for df in best_overall_frames if len(df) > 0], ignore_index=True) if any(len(df) > 0 for df in best_overall_frames) else pd.DataFrame()

saved_regime_exports = []

if len(feasible_both) == 0:
    print('No feasible summaries found for the regime comparison export.')
else:
    strategy_summary = (
        feasible_both
        .groupby(['Window', 'Strategy Mode', 'Cost Regime'], as_index=False)
        .agg({
            'Test Sharpe': 'mean',
            'Cumulative Return': 'mean',
            'Number of Trades': 'mean',
        })
        .rename(columns={
            'Test Sharpe': 'Avg Test Sharpe',
            'Cumulative Return': 'Avg Cumulative Return',
            'Number of Trades': 'Avg Number of Trades',
        })
    )
    positive_share = (
        feasible_both
        .assign(_positive=(feasible_both['Test Sharpe'] > 0).astype(float))
        .groupby(['Window', 'Strategy Mode', 'Cost Regime'], as_index=False)['_positive']
        .mean()
        .rename(columns={'_positive': 'Positive Sharpe Share'})
    )
    strategy_summary = strategy_summary.merge(positive_share, on=['Window', 'Strategy Mode', 'Cost Regime'], how='left')
    saved_regime_exports.append(_save_df(strategy_summary, REGIME_COMPARE_DIR / 'appendix_d_strategy_mode_comparison.csv'))

    family_summary = (
        feasible_both
        .groupby(['Window', 'Model Family', 'Cost Regime'], as_index=False)
        .agg({
            'Test Sharpe': 'mean',
            'Cumulative Return': 'mean',
            'Number of Trades': 'mean',
        })
        .rename(columns={
            'Test Sharpe': 'Avg Test Sharpe',
            'Cumulative Return': 'Avg Cumulative Return',
            'Number of Trades': 'Avg Number of Trades',
        })
    )
    saved_regime_exports.append(_save_df(family_summary, REGIME_COMPARE_DIR / 'appendix_d_model_family_comparison.csv'))

    asset_summary = (
        feasible_both
        .groupby(['Window', 'Asset', 'Cost Regime'], as_index=False)
        .agg({
            'Test Sharpe': 'mean',
            'Cumulative Return': 'mean',
        })
        .rename(columns={
            'Test Sharpe': 'Avg Test Sharpe',
            'Cumulative Return': 'Avg Cumulative Return',
        })
    )
    saved_regime_exports.append(_save_df(asset_summary, REGIME_COMPARE_DIR / 'appendix_d_asset_comparison.csv'))

if len(buy_hold_both) > 0:
    buy_hold_summary = buy_hold_both[[
        'Window', 'Asset', 'Cumulative Return', 'Annualized Return',
        'Annualized Sharpe', 'Maximum Drawdown'
    ]].copy()
    saved_regime_exports.append(_save_df(buy_hold_summary, REGIME_COMPARE_DIR / 'appendix_d_buy_and_hold_comparison.csv'))

if len(group_tests_both) > 0 or len(usefulness_both) > 0:
    sig_rows = []
    for window_label, _ in window_specs:
        group_df = group_tests_both[group_tests_both['Window'] == window_label].copy() if len(group_tests_both) > 0 else pd.DataFrame()
        use_df = usefulness_both[usefulness_both['Window'] == window_label].copy() if len(usefulness_both) > 0 else pd.DataFrame()
        sig_rows.append({
            'Window': window_label,
            'Grouped Tests Significant (p<0.05)': int((group_df['Bootstrap p-value'] < 0.05).sum()) if len(group_df) > 0 else 0,
            'Grouped Tests Total': int(len(group_df)),
            'Usefulness Significant vs 0': int(((use_df['p-value vs 0'] < 0.05) & (use_df['Mean Net Return'] > 0)).sum()) if len(use_df) > 0 else 0,
            'Usefulness Total': int(len(use_df)),
            'Beat Buy-and-Hold Significant': int(((use_df['p-value vs BuyHold'] < 0.05) & (use_df['Mean Excess Return vs BuyHold'] > 0)).sum()) if len(use_df) > 0 else 0,
        })
    significance_summary = pd.DataFrame(sig_rows)
    saved_regime_exports.append(_save_df(significance_summary, REGIME_COMPARE_DIR / 'appendix_d_significance_summary.csv'))

if len(best_overall_both) > 0:
    best_model_counts = (
        best_overall_both.groupby(['Window', 'Model'], as_index=False)
        .size()
        .rename(columns={'size': 'Best Overall Count'})
        .sort_values(['Window', 'Best Overall Count', 'Model'], ascending=[True, False, True])
        .reset_index(drop=True)
    )
    best_feature_counts = (
        best_overall_both.groupby(['Window', 'Feature Set'], as_index=False)
        .size()
        .rename(columns={'size': 'Best Overall Count'})
        .sort_values(['Window', 'Best Overall Count', 'Feature Set'], ascending=[True, False, True])
        .reset_index(drop=True)
    )
    saved_regime_exports.append(_save_df(best_model_counts, REGIME_COMPARE_DIR / 'appendix_d_best_overall_model_counts.csv'))
    saved_regime_exports.append(_save_df(best_feature_counts, REGIME_COMPARE_DIR / 'appendix_d_best_overall_feature_counts.csv'))

plot_outputs = []
if 'strategy_summary' in globals() and len(strategy_summary) > 0:
    for cost_regime in ['no_cost', 'with_cost']:
        subset = strategy_summary[strategy_summary['Cost Regime'] == cost_regime].copy()
        if len(subset) == 0:
            continue
        plot_outputs.append(_plot_grouped_bars(
            subset,
            x_col='Strategy Mode',
            value_col='Avg Test Sharpe',
            hue_col='Window',
            title=f'Average Feasible Test Sharpe by Strategy Mode ({cost_regime.replace("_", " ").title()})',
            ylabel='Average Test Sharpe',
            output_path=REGIME_COMPARE_DIR / f'strategy_mode_test_sharpe_{cost_regime}.png',
            hue_order=['2021-2025', '2020-2024'],
            x_order=['long_only', 'long_short', 'short_only'],
        ))

if 'family_summary' in globals() and len(family_summary) > 0:
    for cost_regime in ['no_cost', 'with_cost']:
        subset = family_summary[family_summary['Cost Regime'] == cost_regime].copy()
        if len(subset) == 0:
            continue
        plot_outputs.append(_plot_grouped_bars(
            subset,
            x_col='Model Family',
            value_col='Avg Test Sharpe',
            hue_col='Window',
            title=f'Average Feasible Test Sharpe by Model Family ({cost_regime.replace("_", " ").title()})',
            ylabel='Average Test Sharpe',
            output_path=REGIME_COMPARE_DIR / f'model_family_test_sharpe_{cost_regime}.png',
            hue_order=['2021-2025', '2020-2024'],
            x_order=['ML', 'Rule'],
        ))

if len(buy_hold_both) > 0:
    plot_outputs.append(_plot_grouped_bars(
        buy_hold_both,
        x_col='Asset',
        value_col='Cumulative Return',
        hue_col='Window',
        title='Buy-and-Hold Cumulative Return by Asset Across Windows',
        ylabel='Cumulative Return',
        output_path=REGIME_COMPARE_DIR / 'buy_and_hold_cumulative_return_comparison.png',
        hue_order=['2021-2025', '2020-2024'],
        x_order=['BTC', 'ETH', 'XRP', 'SOL'],
    ))

plot_outputs = [p for p in plot_outputs if p is not None]
print('Saved regime-comparison tables:')
for path in saved_regime_exports:
    print('-', path)

print('\nSaved regime-comparison figures:')
for path in plot_outputs:
    print('-', path)

Saved regime-comparison tables:
- /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/result/regime_comparison/appendix_c_strategy_mode_comparison.csv
- /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/result/regime_comparison/appendix_c_model_family_comparison.csv
- /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/result/regime_comparison/appendix_c_asset_comparison.csv
- /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/result/regime_comparison/appendix_c_buy_and_hold_comparison.csv
- /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/result/regime_comparison/appendix_c_significance_summary.csv
- /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/result/regime_comparison/appendix_c_best_overall_model_counts.csv
- /Users/wuxuande/PycharmProjects/QF4211Project/2020-2024_regime_sandbox/result/regime_comparison/appendix_c_best_overall_feature_counts.csv

Saved regim